# 01 EDA

Цель ноутбука — провести EDA Step 4 для данных рекомендаций похожих товаров: проверить загрузку и схемы, справочник товаров, структуру `user_actions`, распределение событий по дням и типам, missing values, пользователей, товары, поисковые запросы, sessionization feasibility и готовность данных к preprocessing и первому co-visitation baseline.

Важно: `user_actions` — большой датасет. В EDA он **не материализуется целиком через `.collect()`**. Для анализа используются lazy scan, hive-partitioning, инвентаризация parquet-файлов, partition-level агрегаты, кэшируемые промежуточные таблицы и точечные sample-day diagnostics.


## 1. Imports and paths


In [1]:
from datetime import datetime
from pathlib import Path

import polars as pl
from IPython.display import display


def find_project_root(start: Path) -> Path:
    """Find project root by walking up until pyproject.toml is found."""
    current = start.resolve()

    for path in [current, *current.parents]:
        if (path / "pyproject.toml").exists():
            return path

    raise RuntimeError("Project root was not found. pyproject.toml is missing.")


PROJECT_ROOT = find_project_root(Path.cwd())

PRODUCTS_PATH = PROJECT_ROOT / "data" / "raw" / "product_information"
ACTIONS_PATH = PROJECT_ROOT / "data" / "raw" / "user_actions"
CACHE_DIR = PROJECT_ROOT / "outputs" / "interim"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

FORCE_REBUILD_EDA_CACHE = False

print("Current working directory:", Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("PRODUCTS_PATH exists:", PRODUCTS_PATH.exists(), PRODUCTS_PATH)
print("ACTIONS_PATH exists:", ACTIONS_PATH.exists(), ACTIONS_PATH)

def parquet_filesystem_summary(label: str, root: Path) -> dict:
    parquet_paths = sorted(root.rglob("*.parquet")) if root.exists() else []
    total_size_bytes = sum(path.stat().st_size for path in parquet_paths)
    max_mtime = max((path.stat().st_mtime for path in parquet_paths), default=None)

    return {
        "dataset": label,
        "root": root.relative_to(PROJECT_ROOT).as_posix(),
        "parquet_files": len(parquet_paths),
        "total_size_mb": total_size_bytes / 1024 / 1024,
        "max_mtime": max_mtime,
        "max_modified_at": datetime.fromtimestamp(max_mtime) if max_mtime is not None else None,
    }


def cache_is_stale(path: Path, raw_max_mtime: float | None) -> bool:
    if raw_max_mtime is None or not path.exists():
        return False

    return path.stat().st_mtime < raw_max_mtime


def should_load_eda_cache(path: Path, label: str) -> bool:
    if FORCE_REBUILD_EDA_CACHE:
        print(f"Rebuilding {label}: FORCE_REBUILD_EDA_CACHE=True")
        return False

    if not path.exists():
        return False

    if cache_is_stale(path, RAW_PARQUET_MAX_MTIME):
        cache_mtime = datetime.fromtimestamp(path.stat().st_mtime)
        raw_mtime = datetime.fromtimestamp(RAW_PARQUET_MAX_MTIME)
        raise RuntimeError(
            f"Stale EDA cache for {label}: {path} modified at {cache_mtime}, "
            f"raw parquet max mtime is {raw_mtime}. Delete/rebuild the cache or set "
            "FORCE_REBUILD_EDA_CACHE=True."
        )

    return True


def read_eda_cache(path: Path, label: str) -> pl.DataFrame:
    data = pl.read_parquet(path)
    print(f"Loaded cached {label}: {path}")
    return data


def write_eda_cache(data: pl.DataFrame, path: Path, label: str) -> None:
    data.write_parquet(path)
    print(f"Saved {label} cache: {path}")


EDA_CACHE_PATHS = [
    CACHE_DIR / "eda_rows_by_partition.parquet",
    CACHE_DIR / "eda_partition_timestamp_quality.parquet",
    CACHE_DIR / "eda_user_daily_activity_summary.parquet",
    CACHE_DIR / "eda_item_action_counts_by_partition.parquet",
    CACHE_DIR / "eda_missing_by_partition.parquet",
    CACHE_DIR / "eda_duplicate_check_by_partition.parquet",
    CACHE_DIR / "eda_top_user_by_day.parquet",
    CACHE_DIR / "eda_item_action_counts.parquet",
    CACHE_DIR / "eda_sample_day_item_users_2024-04-24.parquet",
    CACHE_DIR / "eda_search_query_counts.parquet",
]


def cache_filesystem_metadata(path: Path) -> dict:
    exists = path.exists()

    return {
        "path": path.relative_to(PROJECT_ROOT).as_posix(),
        "exists": exists,
        "size_mb": path.stat().st_size / 1024 / 1024 if exists else None,
        "modified_at": datetime.fromtimestamp(path.stat().st_mtime) if exists else None,
        "is_stale_vs_raw": cache_is_stale(path, RAW_PARQUET_MAX_MTIME) if exists else None,
    }


raw_parquet_filesystem_records = [
    parquet_filesystem_summary("product_information", PRODUCTS_PATH),
    parquet_filesystem_summary("user_actions", ACTIONS_PATH),
]
RAW_PARQUET_MAX_MTIME = max(
    (record["max_mtime"] for record in raw_parquet_filesystem_records if record["max_mtime"] is not None),
    default=None,
)

raw_parquet_filesystem_summary = (
    pl.DataFrame(raw_parquet_filesystem_records)
    .drop("max_mtime")
)

eda_cache_filesystem_metadata = pl.DataFrame([
    cache_filesystem_metadata(path)
    for path in EDA_CACHE_PATHS
])

print("Raw parquet filesystem metadata:")
display(raw_parquet_filesystem_summary)

print("EDA cache filesystem metadata:")
display(eda_cache_filesystem_metadata)


Current working directory: C:\Users\User\Projects\OZON-Similar-products\notebooks
PROJECT_ROOT: C:\Users\User\Projects\OZON-Similar-products
PRODUCTS_PATH exists: True C:\Users\User\Projects\OZON-Similar-products\data\raw\product_information
ACTIONS_PATH exists: True C:\Users\User\Projects\OZON-Similar-products\data\raw\user_actions
Raw parquet filesystem metadata:


dataset,root,parquet_files,total_size_mb,max_modified_at
str,str,i64,f64,datetime[μs]
"""product_information""","""data/raw/product_information""",1,4.271684,2026-04-08 19:26:21
"""user_actions""","""data/raw/user_actions""",305,11216.363894,2026-04-09 15:19:39


EDA cache filesystem metadata:


path,exists,size_mb,modified_at,is_stale_vs_raw
str,bool,null,null,null
"""outputs/interim/eda_rows_by_pa…",false,null,null,null
"""outputs/interim/eda_partition_…",false,null,null,null
"""outputs/interim/eda_user_daily…",false,null,null,null
"""outputs/interim/eda_item_actio…",false,null,null,null
"""outputs/interim/eda_missing_by…",false,null,null,null
"""outputs/interim/eda_duplicate_…",false,null,null,null
"""outputs/interim/eda_top_user_b…",false,null,null,null
"""outputs/interim/eda_item_actio…",false,null,null,null
"""outputs/interim/eda_sample_day…",false,null,null,null


In [2]:
product_files = sorted(PRODUCTS_PATH.rglob("*.parquet"))
action_files = sorted(ACTIONS_PATH.rglob("*.parquet"))

print("Product parquet files:", len(product_files))
print("User actions parquet files:", len(action_files))

print("\nProduct files:")
for path in product_files[:5]:
    print("-", path.relative_to(PROJECT_ROOT))

print("\nFirst user action files:")
for path in action_files[:10]:
    print("-", path.relative_to(PROJECT_ROOT))


Product parquet files: 1
User actions parquet files: 305

Product files:
- data\raw\product_information\product_information.parquet

First user action files:
- data\raw\user_actions\user_actions_3_months\date=2024-03-01\action_type=click\part-00000-6b377985-f511-42f3-ae68-09f12a3cc209-c000.snappy.parquet
- data\raw\user_actions\user_actions_3_months\date=2024-03-01\action_type=favorite\part-00000-8457910c-40db-4dd7-83c2-73e80e7add8b-c000.snappy.parquet
- data\raw\user_actions\user_actions_3_months\date=2024-03-01\action_type=search\part-00000-34bba696-5a15-4ab7-9201-5f7fe387e65b-c000.snappy.parquet
- data\raw\user_actions\user_actions_3_months\date=2024-03-01\action_type=to_cart\part-00000-4d4d2fcb-078a-4b72-ae93-137c2f1f0a5c-c000.snappy.parquet
- data\raw\user_actions\user_actions_3_months\date=2024-03-01\action_type=view\part-00000-36b264e2-622b-4bae-9aee-9cf2b65233dd-c000.snappy.parquet
- data\raw\user_actions\user_actions_3_months\date=2024-03-02\action_type=click\part-00000-24766f

## 2. Lazy scans and schema checks


In [3]:
products_lf = pl.scan_parquet(
    str(PRODUCTS_PATH / "**" / "*.parquet")
)

actions_lf = pl.scan_parquet(
    str(ACTIONS_PATH / "**" / "*.parquet"),
    hive_partitioning=True,
)

product_schema = products_lf.collect_schema()
action_schema = actions_lf.collect_schema()

print("Product information schema:")
print(product_schema)

print("\nUser actions schema:")
print(action_schema)


Product information schema:
Schema({'item_id': Int32, 'name': String, 'brand': String, 'type': String, 'category_id': Int32, 'category_name': String})

User actions schema:
Schema({'user_id': Int32, 'date': Date, 'timestamp': Datetime(time_unit='ns', time_zone=None), 'action_type': String, 'widget_name': String, 'search_query': String, 'item_id': Int32})


In [4]:
expected_product_columns = [
    "item_id",
    "name",
    "brand",
    "type",
    "category_id",
    "category_name",
]

expected_action_columns = [
    "user_id",
    "date",
    "timestamp",
    "action_type",
    "widget_name",
    "search_query",
    "item_id",
]


def build_schema_check(expected_columns: list[str], schema: pl.Schema) -> pl.DataFrame:
    actual_columns = list(schema.names())

    return pl.DataFrame({
        "expected_column": expected_columns,
        "is_present": [column in actual_columns for column in expected_columns],
        "actual_dtype": [
            str(schema[column]) if column in actual_columns else None
            for column in expected_columns
        ],
    })


product_column_check = build_schema_check(expected_product_columns, product_schema)
action_column_check = build_schema_check(expected_action_columns, action_schema)

print("Product column check:")
display(product_column_check)

print("User actions column check:")
display(action_column_check)


Product column check:


expected_column,is_present,actual_dtype
str,bool,str
"""item_id""",true,"""Int32"""
"""name""",true,"""String"""
"""brand""",true,"""String"""
"""type""",true,"""String"""
"""category_id""",true,"""Int32"""
"""category_name""",true,"""String"""


User actions column check:


expected_column,is_present,actual_dtype
str,bool,str
"""user_id""",true,"""Int32"""
"""date""",true,"""Date"""
"""timestamp""",true,"""Datetime(time_unit='ns', time_…"
"""action_type""",true,"""String"""
"""widget_name""",true,"""String"""
"""search_query""",true,"""String"""
"""item_id""",true,"""Int32"""


In [5]:
products_rows = products_lf.select(pl.len().alias("rows")).collect()
actions_rows = actions_lf.select(pl.len().alias("rows")).collect()

print("Product information rows:")
display(products_rows)

print("User actions rows:")
display(actions_rows)


Product information rows:


rows
u32
130035


User actions rows:


rows
u32
561836992


In [6]:
products_sample = products_lf.head(5).collect()
actions_sample = actions_lf.head(10).collect()

print("Products sample:")
display(products_sample)

print("User actions sample:")
display(actions_sample)


Products sample:


item_id,name,brand,type,category_id,category_name
i32,str,str,str,i32,str
12291,"""Витамин С 1200 мг Эвалар, для …","""Эвалар""","""Витаминный комплекс""",761,"""Витамин C"""
178066,"""Nature's Bounty Эстер-С 500мг …","""Nature's Bounty""","""Витамины""",761,"""Витамин C"""
169337,"""Доппельгерц Актив Витамин C + …","""Доппельгерц / Doppelherz""","""Витаминный комплекс""",761,"""Витамин C"""
226982,"""Витаминный комплекс Бэби Форму…","""Эвалар""","""Витаминный комплекс""",761,"""Витамин C"""
220079,"""Maxler Vitamin C (Sodium Ascor…","""Maxler""","""Витамины""",761,"""Витамин C"""


User actions sample:


user_id,date,timestamp,action_type,widget_name,search_query,item_id
i32,date,datetime[ns],str,str,str,i32
3945998,2024-03-01,2024-03-01 07:59:07,"""click""","""search_catalog_listing""","""супрадин кидс""",233485
9021968,2024-03-01,2024-03-01 14:59:01,"""click""","""search_catalog_listing""","""протеин""",136372
5332297,2024-03-01,2024-03-01 08:18:23,"""click""","""search_catalog_listing""","""макароны""",208422
10585880,2024-03-01,2024-03-01 09:20:30,"""click""","""search_catalog_listing""","""прокладки женские""",216302
1405707,2024-03-01,2024-03-01 08:55:25,"""click""","""search_catalog_listing""","""фрикадельки""",57312
6382797,2024-03-01,2024-03-01 14:51:01,"""click""","""search_catalog_listing""","""десерт""",216504
5415873,2024-03-01,2024-03-01 22:25:00,"""click""","""search_catalog_listing""","""аргентинские креветки""",221484
4059064,2024-03-01,2024-03-01 05:00:24,"""click""","""search_catalog_listing""","""пробные линзы""",237157
8613176,2024-03-01,2024-03-01 06:57:20,"""click""","""search_catalog_listing""","""wokali маска""",186620


## 3. Product information overview


In [7]:
products = products_lf.collect()

print("products shape:", products.shape)

products_overview = products.select(
    pl.len().alias("rows"),
    pl.col("item_id").n_unique().alias("unique_item_id"),
    (pl.len() - pl.col("item_id").n_unique()).alias("duplicate_item_id_count"),
    pl.col("category_id").n_unique().alias("unique_category_id"),
    pl.col("category_name").n_unique().alias("unique_category_name"),
    pl.col("brand").n_unique().alias("unique_brand"),
    pl.col("type").n_unique().alias("unique_type"),
)

display(products_overview)


products shape: (130035, 6)


rows,unique_item_id,duplicate_item_id_count,unique_category_id,unique_category_name,unique_brand,unique_type
u32,u32,u32,u32,u32,u32,u32
130035,130035,0,834,796,6682,2184


In [8]:
product_nulls = (
    products
    .null_count()
    .transpose(
        include_header=True,
        header_name="column",
        column_names=["null_count"],
    )
    .with_columns(
        (pl.col("null_count") / products.height).alias("null_share")
    )
    .sort("null_share", descending=True)
)

display(product_nulls)


column,null_count,null_share
str,u32,f64
"""brand""",227,0.001746
"""category_name""",89,0.000684
"""item_id""",0,0.0
"""name""",0,0.0
"""type""",0,0.0
"""category_id""",0,0.0


In [9]:
top_categories = (
    products
    .group_by(["category_id", "category_name"])
    .agg(pl.len().alias("products_count"))
    .sort("products_count", descending=True)
    .head(20)
)

top_brands = (
    products
    .group_by("brand")
    .agg(pl.len().alias("products_count"))
    .sort("products_count", descending=True)
    .head(20)
)

top_types = (
    products
    .group_by("type")
    .agg(pl.len().alias("products_count"))
    .sort("products_count", descending=True)
    .head(20)
)

print("Top categories:")
display(top_categories)

print("Top brands:")
display(top_brands)

print("Top types:")
display(top_types)


Top categories:


category_id,category_name,products_count
i32,str,u32
54,"""Наполнители""",7698
445,"""Сухие корма""",3287
347,"""Для цветного белья""",3142
754,"""Сухие корма""",2954
771,"""Влажные корма""",2515
…,…,…
650,"""Зубные пасты и ополаскиватели""",1064
165,"""Кабели и адаптеры""",1054
85,"""Спортивные батончики и печенье""",1044


Top brands:


brand,products_count
str,u32
"""PET PRIDE""",4684
"""Нет бренда""",2774
"""Tide""",2049
"""LEGO""",1978
"""PRO PLAN""",1731
…,…
"""KIX""",826
"""Homecat""",816
"""Ollin Professional""",778


Top types:


type,products_count
str,u32
"""Наполнитель""",7721
"""Корм сухой""",6285
"""Стиральный порошок""",4805
"""Корм влажный""",2940
"""Зубная паста""",2711
…,…
"""Жидкое средство для стирки""",1064
"""Печатная книга""",1054
"""Пюре""",1029


In [10]:
products_with_missing_metadata = (
    products
    .filter(
        pl.col("name").is_null()
        | pl.col("brand").is_null()
        | pl.col("type").is_null()
        | pl.col("category_id").is_null()
        | pl.col("category_name").is_null()
    )
)

brand_quality = products.select(
    pl.len().alias("rows"),
    pl.col("brand").is_null().sum().alias("brand_null_count"),
    (pl.col("brand") == "Нет бренда").sum().alias("brand_no_brand_count"),
    (
        pl.col("brand").is_null()
        | (pl.col("brand") == "Нет бренда")
    ).sum().alias("brand_unknown_count"),
).with_columns(
    (pl.col("brand_unknown_count") / pl.col("rows")).alias("brand_unknown_share")
)

print("Products with any missing metadata:", products_with_missing_metadata.height)
display(products_with_missing_metadata.head(20))
display(brand_quality)


Products with any missing metadata: 316


item_id,name,brand,type,category_id,category_name
i32,str,str,str,i32,str
233818,"""Внешняя звуковая карта USB для…","""NATION PRIDE""","""Звуковая карта""",811,null
244596,"""Электронный модуль Мастер Кит …","""Мастер Кит""","""Промышленная панель управления""",811,null
203795,"""Силиконовая смазка для компьют…","""STEEL""","""Смазка для компьютерных вентил…",811,null
62398,"""Термопрокладка теплопроводящая…","""3kS""","""Термопрокладка""",811,null
55611,"""Кофе в капсулах Tassimo Bailey…","""Tassimo""","""Кофе в капсулах""",185,null
…,…,…,…,…,…
141186,"""Набор кофе капсульного Tassimo…","""Tassimo""","""Кофе в капсулах""",185,null
26034,"""Кофе в капсулах Tassimo Jacobs…","""Jacobs""","""Кофе в капсулах""",185,null
226660,"""Кофе капсульный Tassimo L'OR X…","""Tassimo""","""Кофе в капсулах""",185,null


rows,brand_null_count,brand_no_brand_count,brand_unknown_count,brand_unknown_share
u32,u32,u32,u32,f64
130035,227,2774,3001,0.023078


In [11]:
NO_BRAND_LABEL = "\u041d\u0435\u0442 \u0431\u0440\u0435\u043d\u0434\u0430"

product_metadata_guardrail = products.select([
    pl.len().alias("products"),
    pl.col("item_id").n_unique().alias("unique_item_id"),
    (pl.len() - pl.col("item_id").n_unique()).alias("duplicate_item_id_count"),
    pl.col("brand").is_null().sum().alias("brand_null_count"),
    (pl.col("brand") == NO_BRAND_LABEL).sum().alias("brand_no_brand_count"),
    (
        pl.col("brand").is_null()
        | (pl.col("brand") == NO_BRAND_LABEL)
    ).sum().alias("brand_unknown_count"),
    pl.col("category_id").is_null().sum().alias("category_id_null_count"),
    pl.col("category_name").is_null().sum().alias("category_name_null_count"),
    (
        pl.col("category_name").is_null()
        & pl.col("category_id").is_not_null()
    ).sum().alias("category_name_null_with_category_id_count"),
]).with_columns([
    (pl.col("brand_unknown_count") / pl.col("products")).alias("brand_unknown_share"),
    (pl.col("category_name_null_count") / pl.col("products")).alias("category_name_null_share"),
])

print("Product metadata guardrail checks:")
display(product_metadata_guardrail)


Product metadata guardrail checks:


products,unique_item_id,duplicate_item_id_count,brand_null_count,brand_no_brand_count,brand_unknown_count,category_id_null_count,category_name_null_count,category_name_null_with_category_id_count,brand_unknown_share,category_name_null_share
u32,u32,u32,u32,u32,u32,u32,u32,u32,f64,f64
130035,130035,0,227,2774,3001,0,89,89,0.023078,0.000684


In [12]:
category_name_to_ids = (
    products
    .filter(pl.col("category_name").is_not_null())
    .group_by("category_name")
    .agg([
        pl.col("category_id").n_unique().alias("category_id_count"),
        pl.col("category_id").unique().sort().alias("category_ids"),
        pl.len().alias("products_count"),
    ])
    .filter(pl.col("category_id_count") > 1)
    .sort(["category_id_count", "products_count"], descending=[True, True])
)

print("Category names mapped to more than one category_id:", category_name_to_ids.height)
display(category_name_to_ids.head(20))


Category names mapped to more than one category_id: 25


category_name,category_id_count,category_ids,products_count
str,u32,list[i32],u32
"""Аксессуары""",4,"[15, 26, … 204]",675
"""Печенье""",3,"[98, 589, 605]",591
"""Грибы""",3,"[89, 279, 570]",64
"""Сухие корма""",2,"[445, 754]",6241
"""Влажные корма""",2,"[31, 771]",2939
…,…,…,…
"""Овощные""",2,"[328, 466]",101
"""Противогрибковые""",2,"[216, 808]",88
"""Батончики""",2,"[756, 772]",68


### Product information conclusions

- Справочник содержит 130 035 товаров.
- `item_id` уникален: дублей нет.
- `category_id` заполнен полностью и подходит как основной идентификатор категории для fallback-логики внутри текущего датасета.
- `category_name` имеет небольшое число пропусков, но `category_id` при этом заполнен, поэтому проблема не критична для baseline.
- `brand` имеет null-пропуски; дополнительно есть текстовое значение `"Нет бренда"`. Для анализа и business rules такие значения нужно трактовать как неизвестный бренд.
- `category_name` не является уникальным идентификатором категории: одно название может соответствовать разным `category_id`.
- Для baseline и fallback на текущем этапе лучше использовать `category_id`; для ручной интерпретации результатов — `category_name`, `name`, `brand`, `type`.


## 4. User actions file inventory


In [13]:
file_inventory = pl.DataFrame({
    "path": [path.as_posix() for path in action_files],
    "relative_path": [path.relative_to(PROJECT_ROOT).as_posix() for path in action_files],
    "size_mb": [path.stat().st_size / 1024 / 1024 for path in action_files],
})

file_inventory = file_inventory.with_columns(
    pl.col("relative_path").str.extract(r"date=(\d{4}-\d{2}-\d{2})", 1).alias("date"),
    pl.col("relative_path").str.extract(r"action_type=([^/]+)", 1).alias("action_type"),
)

file_inventory = file_inventory.with_columns(
    pl.col("date").str.strptime(pl.Date, "%Y-%m-%d")
)

display(file_inventory.head(10))

file_inventory_overview = file_inventory.select(
    pl.len().alias("parquet_files"),
    pl.col("date").min().alias("min_partition_date"),
    pl.col("date").max().alias("max_partition_date"),
    pl.col("date").n_unique().alias("partition_days"),
    pl.col("action_type").n_unique().alias("partition_action_types"),
    pl.col("size_mb").sum().alias("total_size_mb"),
)

display(file_inventory_overview)


path,relative_path,size_mb,date,action_type
str,str,f64,date,str
"""C:/Users/User/Projects/OZON-Si…","""data/raw/user_actions/user_act…",4.607059,2024-03-01,"""click"""
"""C:/Users/User/Projects/OZON-Si…","""data/raw/user_actions/user_act…",0.259104,2024-03-01,"""favorite"""
"""C:/Users/User/Projects/OZON-Si…","""data/raw/user_actions/user_act…",12.670323,2024-03-01,"""search"""
"""C:/Users/User/Projects/OZON-Si…","""data/raw/user_actions/user_act…",5.168841,2024-03-01,"""to_cart"""
"""C:/Users/User/Projects/OZON-Si…","""data/raw/user_actions/user_act…",190.177905,2024-03-01,"""view"""
"""C:/Users/User/Projects/OZON-Si…","""data/raw/user_actions/user_act…",5.151161,2024-03-02,"""click"""
"""C:/Users/User/Projects/OZON-Si…","""data/raw/user_actions/user_act…",0.256618,2024-03-02,"""favorite"""
"""C:/Users/User/Projects/OZON-Si…","""data/raw/user_actions/user_act…",13.676371,2024-03-02,"""search"""
"""C:/Users/User/Projects/OZON-Si…","""data/raw/user_actions/user_act…",5.720272,2024-03-02,"""to_cart"""


parquet_files,min_partition_date,max_partition_date,partition_days,partition_action_types,total_size_mb
u32,date,date,u32,u32,f64
305,2024-03-01,2024-04-30,61,5,11216.363894


In [14]:
files_by_day = (
    file_inventory
    .group_by("date")
    .agg(
        pl.len().alias("parquet_files"),
        pl.col("action_type").n_unique().alias("action_types"),
        pl.col("size_mb").sum().alias("size_mb"),
    )
    .sort("date")
)

files_by_action_type = (
    file_inventory
    .group_by("action_type")
    .agg(
        pl.len().alias("parquet_files"),
        pl.col("date").n_unique().alias("days"),
        pl.col("size_mb").sum().alias("size_mb"),
    )
    .sort("size_mb", descending=True)
)

display(files_by_day)
display(files_by_action_type)


date,parquet_files,action_types,size_mb
date,u32,u32,f64
2024-03-01,5,5,212.883232
2024-03-02,5,5,232.820042
2024-03-03,5,5,227.423747
2024-03-04,5,5,198.996922
2024-03-05,5,5,222.475039
…,…,…,…
2024-04-26,5,5,157.273384
2024-04-27,5,5,188.11931
2024-04-28,5,5,175.897675


action_type,parquet_files,days,size_mb
str,u32,u32,f64
"""view""",61,61,10026.76433
"""search""",61,61,682.219216
"""to_cart""",61,61,268.73946
"""click""",61,61,226.028446
"""favorite""",61,61,12.612441


In [15]:
smallest_day = (
    files_by_day
    .sort("size_mb")
    .get_column("date")
    .first()
)

print("Smallest day:", smallest_day)

sample_day_files = (
    file_inventory
    .filter(pl.col("date") == smallest_day)
    .get_column("path")
    .to_list()
)

sample_day_lf = pl.scan_parquet(
    sample_day_files,
    hive_partitioning=True,
)

sample_day_overview = sample_day_lf.select(
    pl.len().alias("rows"),
    pl.col("date").min().alias("min_date"),
    pl.col("date").max().alias("max_date"),
    pl.col("timestamp").min().alias("min_timestamp"),
    pl.col("timestamp").max().alias("max_timestamp"),
).collect()

display(sample_day_overview)


Smallest day: 2024-04-24


rows,min_date,max_date,min_timestamp,max_timestamp
u32,date,date,datetime[ns],datetime[ns]
7431439,2024-04-24,2024-04-24,2024-04-24 00:00:00,2024-04-24 23:59:59


### User actions file inventory conclusions

- `user_actions` физически хранится как набор parquet-файлов в hive-структуре по `date` и `action_type`.
- Структура файлов полная: для каждого дня есть все 5 ожидаемых партиций по типам событий.
- Общий размер parquet-файлов около 11.2 GB, поэтому полный датасет не собирается в память на EDA-этапе.
- Самая тяжелая по объему партиция — `view`; это ограничивает выбор EDA-операций, поэтому предпочтительны partition-level и cached агрегаты.


## 5. Rows by partition, by day, and daily pipeline readiness

В этом блоке считаются только partition-level и cached агрегаты. Глобального чтения `user_actions` целиком нет.

Важно: EDA-кеши в `outputs/interim` используются для ускорения анализа. При чтении кешей notebook проверяет, что cache `mtime` не старше максимального `mtime` raw parquet; stale cache блокирует выполнение. Финальное состояние кешей нужно смотреть в post-run freshness table после всех cache-producing блоков.


In [16]:
ROWS_BY_PARTITION_PATH = CACHE_DIR / "eda_rows_by_partition.parquet"

if should_load_eda_cache(ROWS_BY_PARTITION_PATH, "rows_by_partition"):
    rows_by_partition = read_eda_cache(ROWS_BY_PARTITION_PATH, "rows_by_partition")
else:
    row_count_records = []

    for i, row in enumerate(file_inventory.iter_rows(named=True), start=1):
        rows_count = (
            pl.scan_parquet(row["path"])
            .select(pl.len().alias("rows"))
            .collect()
            .item()
        )

        row_count_records.append({
            "date": row["date"],
            "action_type": row["action_type"],
            "rows": rows_count,
            "size_mb": row["size_mb"],
        })

        if i % 25 == 0:
            print(f"Processed {i} / {file_inventory.height} files")

    rows_by_partition = pl.DataFrame(row_count_records)
    write_eda_cache(rows_by_partition, ROWS_BY_PARTITION_PATH, "rows_by_partition")

display(rows_by_partition.head(10))


Processed 25 / 305 files
Processed 50 / 305 files
Processed 75 / 305 files
Processed 100 / 305 files
Processed 125 / 305 files
Processed 150 / 305 files
Processed 175 / 305 files
Processed 200 / 305 files
Processed 225 / 305 files
Processed 250 / 305 files
Processed 275 / 305 files
Processed 300 / 305 files
Saved rows_by_partition cache: C:\Users\User\Projects\OZON-Similar-products\outputs\interim\eda_rows_by_partition.parquet


date,action_type,rows,size_mb
date,str,i64,f64
2024-03-01,"""click""",230035,4.607059
2024-03-01,"""favorite""",10242,0.259104
2024-03-01,"""search""",592151,12.670323
2024-03-01,"""to_cart""",333478,5.168841
2024-03-01,"""view""",9280153,190.177905
2024-03-02,"""click""",254477,5.151161
2024-03-02,"""favorite""",9847,0.256618
2024-03-02,"""search""",642899,13.676371
2024-03-02,"""to_cart""",361499,5.720272


In [17]:
daily_rows = (
    rows_by_partition
    .group_by("date")
    .agg(
        pl.col("rows").sum().alias("rows"),
        pl.col("size_mb").sum().alias("size_mb"),
        pl.when(pl.col("action_type") == "search").then(pl.col("rows")).otherwise(0).sum().alias("search"),
        pl.when(pl.col("action_type") == "view").then(pl.col("rows")).otherwise(0).sum().alias("view"),
        pl.when(pl.col("action_type") == "click").then(pl.col("rows")).otherwise(0).sum().alias("click"),
        pl.when(pl.col("action_type") == "favorite").then(pl.col("rows")).otherwise(0).sum().alias("favorite"),
        pl.when(pl.col("action_type") == "to_cart").then(pl.col("rows")).otherwise(0).sum().alias("to_cart"),
    )
    .sort("date")
)

action_type_rows = (
    rows_by_partition
    .group_by("action_type")
    .agg(
        pl.col("rows").sum().alias("rows"),
        pl.col("size_mb").sum().alias("size_mb"),
        pl.col("date").n_unique().alias("days"),
    )
    .with_columns(
        (pl.col("rows") / pl.col("rows").sum()).alias("share")
    )
    .sort("rows", descending=True)
)

display(daily_rows)
display(action_type_rows)


date,rows,size_mb,search,view,click,favorite,to_cart
date,i64,f64,i64,i64,i64,i64,i64
2024-03-01,10446059,212.883232,592151,9280153,230035,10242,333478
2024-03-02,11454573,232.820042,642899,10185851,254477,9847,361499
2024-03-03,11127161,227.423747,624602,9899880,245688,10263,346728
2024-03-04,9638934,198.996922,538250,8589520,212654,9699,288811
2024-03-05,10859997,222.475039,589928,9708096,238194,10410,313369
…,…,…,…,…,…,…,…
2024-04-26,7957682,157.273384,519806,6992884,174672,7159,263161
2024-04-27,9534450,188.11931,560453,8434003,210528,7692,321774
2024-04-28,8831496,175.897675,526365,7810240,199960,7446,287485


action_type,rows,size_mb,days,share
str,i64,f64,u32,f64
"""view""",498735607,10026.76433,61,0.887687
"""search""",32274194,682.219216,61,0.057444
"""to_cart""",18110504,268.73946,61,0.032234
"""click""",12224896,226.028446,61,0.021759
"""favorite""",491791,12.612441,61,0.000875


In [18]:
daily_rows_summary = daily_rows.select(
    pl.len().alias("days"),
    pl.col("rows").min().alias("min_rows"),
    pl.col("rows").max().alias("max_rows"),
    pl.col("rows").mean().alias("mean_rows"),
    pl.col("rows").median().alias("median_rows"),
    pl.col("rows").quantile(0.25).alias("p25_rows"),
    pl.col("rows").quantile(0.75).alias("p75_rows"),
)

display(daily_rows_summary)

print("Days with the fewest events:")
display(daily_rows.sort("rows").head(10))

print("Days with the most events:")
display(daily_rows.sort("rows", descending=True).head(10))


days,min_rows,max_rows,mean_rows,median_rows,p25_rows,p75_rows
u32,i64,i64,f64,f64,f64,f64
61,7431439,12215617,9.2104e6,9.069222e6,8.454958e6,9.638934e6


Days with the fewest events:


date,rows,size_mb,search,view,click,favorite,to_cart
date,i64,f64,i64,i64,i64,i64,i64
2024-04-24,7431439,147.45398,441598,6569481,163953,6969,249438
2024-04-23,7592262,150.59677,445181,6718112,168758,6944,253267
2024-04-08,7828683,155.327081,450011,6947215,167310,6958,257189
2024-04-10,7831804,154.861646,449688,6951742,164725,6945,258704
2024-04-22,7850369,156.223449,462803,6942754,174683,6974,263155
2024-04-01,7899805,158.749285,472161,6986823,169962,7236,263623
2024-04-11,7940761,157.974888,459727,7039936,171080,6673,263345
2024-04-16,7945123,157.566855,457825,7046835,172085,6652,261726
2024-04-09,7951447,156.972337,454624,7059680,168382,7064,261697


Days with the most events:


date,rows,size_mb,search,view,click,favorite,to_cart
date,i64,f64,i64,i64,i64,i64,i64
2024-03-07,12215617,240.220968,648089,10954461,261137,10112,341818
2024-03-23,11638705,230.407466,657729,10341289,250257,9353,380077
2024-03-06,11517660,227.178148,607499,10332248,244966,10370,322577
2024-03-02,11454573,232.820042,642899,10185851,254477,9847,361499
2024-03-24,11148168,224.223432,637481,9906665,244382,9234,350406
2024-03-03,11127161,227.423747,624602,9899880,245688,10263,346728
2024-03-08,10946358,213.68527,594844,9802584,233491,8684,306755
2024-03-05,10859997,222.475039,589928,9708096,238194,10410,313369
2024-03-10,10778894,216.578464,604444,9584222,233575,8226,348427


In [19]:
PARTITION_TIMESTAMP_QUALITY_PATH = CACHE_DIR / "eda_partition_timestamp_quality.parquet"

if should_load_eda_cache(PARTITION_TIMESTAMP_QUALITY_PATH, "partition_timestamp_quality"):
    partition_timestamp_quality = read_eda_cache(PARTITION_TIMESTAMP_QUALITY_PATH, "partition_timestamp_quality")
else:
    timestamp_quality_records = []

    for i, row in enumerate(file_inventory.iter_rows(named=True), start=1):
        partition_quality = (
            pl.scan_parquet(row["path"], hive_partitioning=True)
            .select([
                pl.len().alias("rows"),
                pl.col("timestamp").min().alias("min_timestamp"),
                pl.col("timestamp").max().alias("max_timestamp"),
                (
                    pl.col("timestamp").dt.date() != pl.col("date")
                ).sum().alias("date_timestamp_mismatch_count"),
            ])
            .collect()
        )

        record = partition_quality.to_dicts()[0]
        record.update({
            "date": row["date"],
            "action_type": row["action_type"],
            "size_mb": row["size_mb"],
            "relative_path": row["relative_path"],
        })
        timestamp_quality_records.append(record)

        if i % 25 == 0:
            print(f"Processed {i} / {file_inventory.height} files")

    partition_timestamp_quality = pl.DataFrame(timestamp_quality_records)
    write_eda_cache(partition_timestamp_quality, PARTITION_TIMESTAMP_QUALITY_PATH, "partition_timestamp_quality")

daily_timestamp_quality = (
    partition_timestamp_quality
    .group_by("date")
    .agg([
        pl.col("min_timestamp").min().alias("min_timestamp"),
        pl.col("max_timestamp").max().alias("max_timestamp"),
        pl.col("date_timestamp_mismatch_count").sum().alias("date_timestamp_mismatch_count"),
    ])
    .sort("date")
)

display(daily_timestamp_quality)


Processed 25 / 305 files
Processed 50 / 305 files
Processed 75 / 305 files
Processed 100 / 305 files
Processed 125 / 305 files
Processed 150 / 305 files
Processed 175 / 305 files
Processed 200 / 305 files
Processed 225 / 305 files
Processed 250 / 305 files
Processed 275 / 305 files
Processed 300 / 305 files
Saved partition_timestamp_quality cache: C:\Users\User\Projects\OZON-Similar-products\outputs\interim\eda_partition_timestamp_quality.parquet


date,min_timestamp,max_timestamp,date_timestamp_mismatch_count
date,datetime[μs],datetime[μs],i64
2024-03-01,2024-03-01 00:00:00,2024-03-01 23:59:59,0
2024-03-02,2024-03-02 00:00:00,2024-03-02 23:59:59,0
2024-03-03,2024-03-03 00:00:00,2024-03-03 23:59:59,0
2024-03-04,2024-03-04 00:00:00,2024-03-04 23:59:59,0
2024-03-05,2024-03-05 00:00:00,2024-03-05 23:59:59,0
…,…,…,…
2024-04-26,2024-04-26 00:00:00,2024-04-26 23:59:59,0
2024-04-27,2024-04-27 00:00:00,2024-04-27 23:59:59,0
2024-04-28,2024-04-28 00:00:00,2024-04-28 23:59:59,0


In [20]:
USER_DAILY_ACTIVITY_SUMMARY_PATH = CACHE_DIR / "eda_user_daily_activity_summary.parquet"

if should_load_eda_cache(USER_DAILY_ACTIVITY_SUMMARY_PATH, "user_daily_activity_summary"):
    user_daily_activity_summary = read_eda_cache(USER_DAILY_ACTIVITY_SUMMARY_PATH, "user_daily_activity_summary")
else:
    user_daily_summary_records = []

    for i, day in enumerate(daily_rows.get_column("date").to_list(), start=1):
        day_files = (
            file_inventory
            .filter(pl.col("date") == day)
            .get_column("path")
            .to_list()
        )

        day_lf = pl.scan_parquet(day_files, hive_partitioning=True)

        user_activity_day = (
            day_lf
            .group_by("user_id")
            .agg([
                pl.len().alias("events"),
                pl.col("item_id").drop_nulls().n_unique().alias("unique_items"),
            ])
            .collect()
        )

        day_summary = user_activity_day.select([
            pl.len().alias("users"),
            pl.col("events").sum().alias("events"),
            pl.col("events").mean().alias("mean_events_per_user"),
            pl.col("events").median().alias("median_events_per_user"),
            pl.col("events").quantile(0.90).alias("p90_events_per_user"),
            pl.col("events").quantile(0.95).alias("p95_events_per_user"),
            pl.col("events").quantile(0.99).alias("p99_events_per_user"),
            pl.col("events").max().alias("max_events_per_user"),
            pl.col("unique_items").median().alias("median_unique_items_per_user"),
            pl.col("unique_items").quantile(0.99).alias("p99_unique_items_per_user"),
            pl.col("unique_items").max().alias("max_unique_items_per_user"),
        ]).with_columns(
            pl.lit(day).alias("date")
        )

        user_daily_summary_records.append(day_summary)

        print(f"Processed {i} / {daily_rows.height} days: {day}")

    user_daily_activity_summary = pl.concat(user_daily_summary_records).select([
        "date",
        "users",
        "events",
        "mean_events_per_user",
        "median_events_per_user",
        "p90_events_per_user",
        "p95_events_per_user",
        "p99_events_per_user",
        "max_events_per_user",
        "median_unique_items_per_user",
        "p99_unique_items_per_user",
        "max_unique_items_per_user",
    ])

    write_eda_cache(user_daily_activity_summary, USER_DAILY_ACTIVITY_SUMMARY_PATH, "user_daily_activity_summary")

display(user_daily_activity_summary.head())


Processed 1 / 61 days: 2024-03-01
Processed 2 / 61 days: 2024-03-02
Processed 3 / 61 days: 2024-03-03
Processed 4 / 61 days: 2024-03-04
Processed 5 / 61 days: 2024-03-05
Processed 6 / 61 days: 2024-03-06
Processed 7 / 61 days: 2024-03-07
Processed 8 / 61 days: 2024-03-08
Processed 9 / 61 days: 2024-03-09
Processed 10 / 61 days: 2024-03-10
Processed 11 / 61 days: 2024-03-11
Processed 12 / 61 days: 2024-03-12
Processed 13 / 61 days: 2024-03-13
Processed 14 / 61 days: 2024-03-14
Processed 15 / 61 days: 2024-03-15
Processed 16 / 61 days: 2024-03-16
Processed 17 / 61 days: 2024-03-17
Processed 18 / 61 days: 2024-03-18
Processed 19 / 61 days: 2024-03-19
Processed 20 / 61 days: 2024-03-20
Processed 21 / 61 days: 2024-03-21
Processed 22 / 61 days: 2024-03-22
Processed 23 / 61 days: 2024-03-23
Processed 24 / 61 days: 2024-03-24
Processed 25 / 61 days: 2024-03-25
Processed 26 / 61 days: 2024-03-26
Processed 27 / 61 days: 2024-03-27
Processed 28 / 61 days: 2024-03-28
Processed 29 / 61 days: 2024-

date,users,events,mean_events_per_user,median_events_per_user,p90_events_per_user,p95_events_per_user,p99_events_per_user,max_events_per_user,median_unique_items_per_user,p99_unique_items_per_user,max_unique_items_per_user
date,u32,u32,f64,f64,f64,f64,f64,u32,f64,f64,u32
2024-03-01,164133,10446059,63.643868,25.0,162.0,241.0,465.0,438686,18.0,339.0,30931
2024-03-02,174892,11454573,65.495123,25.0,167.0,250.0,475.0,508144,18.0,347.0,30736
2024-03-03,177518,11127161,62.681875,24.0,160.0,240.0,453.0,530910,17.0,327.0,30715
2024-03-04,166176,9638934,58.004369,22.0,146.0,220.0,428.0,496340,15.0,309.0,30597
2024-03-05,178864,10859997,60.716505,23.0,154.0,232.0,448.0,515495,16.0,321.0,31022


In [21]:
ITEM_ACTION_COUNTS_BY_PARTITION_PATH = CACHE_DIR / "eda_item_action_counts_by_partition.parquet"
item_action_types = ["view", "click", "favorite", "to_cart"]

if should_load_eda_cache(ITEM_ACTION_COUNTS_BY_PARTITION_PATH, "item_action_counts_by_partition"):
    item_action_counts_by_partition = read_eda_cache(ITEM_ACTION_COUNTS_BY_PARTITION_PATH, "item_action_counts_by_partition")
else:
    item_count_frames = []

    item_files = (
        file_inventory
        .filter(pl.col("action_type").is_in(item_action_types))
        .sort(["date", "action_type"])
    )

    for i, row in enumerate(item_files.iter_rows(named=True), start=1):
        item_counts_partition = (
            pl.scan_parquet(row["path"], hive_partitioning=True)
            .filter(pl.col("item_id").is_not_null())
            .group_by("item_id")
            .agg(pl.len().alias("events"))
            .with_columns([
                pl.lit(row["date"]).alias("date"),
                pl.lit(row["action_type"]).alias("action_type"),
            ])
            .select(["date", "action_type", "item_id", "events"])
            .collect(engine="streaming")
        )

        item_count_frames.append(item_counts_partition)

        if i % 25 == 0:
            print(f"Processed {i} / {item_files.height} item partitions")

    item_action_counts_by_partition = pl.concat(item_count_frames)
    write_eda_cache(item_action_counts_by_partition, ITEM_ACTION_COUNTS_BY_PARTITION_PATH, "item_action_counts_by_partition")

display(item_action_counts_by_partition.head(10))


Processed 25 / 244 item partitions
Processed 50 / 244 item partitions
Processed 75 / 244 item partitions
Processed 100 / 244 item partitions
Processed 125 / 244 item partitions
Processed 150 / 244 item partitions
Processed 175 / 244 item partitions
Processed 200 / 244 item partitions
Processed 225 / 244 item partitions
Saved item_action_counts_by_partition cache: C:\Users\User\Projects\OZON-Similar-products\outputs\interim\eda_item_action_counts_by_partition.parquet


date,action_type,item_id,events
date,str,i32,u32
2024-03-01,"""click""",208422,149
2024-03-01,"""click""",164849,120
2024-03-01,"""click""",175741,11
2024-03-01,"""click""",173973,82
2024-03-01,"""click""",200980,66
2024-03-01,"""click""",70743,1
2024-03-01,"""click""",34177,8
2024-03-01,"""click""",136341,11
2024-03-01,"""click""",131009,21


In [22]:
daily_items = (
    item_action_counts_by_partition
    .select(["date", "item_id"])
    .unique()
    .group_by("date")
    .agg(pl.len().alias("items"))
    .sort("date")
)

daily_rows_enriched = (
    daily_rows
    .join(user_daily_activity_summary.select(["date", "users"]), on="date", how="left")
    .join(daily_items, on="date", how="left")
    .join(daily_timestamp_quality, on="date", how="left")
    .select([
        "date",
        "rows",
        "users",
        "items",
        "search",
        "view",
        "click",
        "favorite",
        "to_cart",
        "size_mb",
        "min_timestamp",
        "max_timestamp",
        "date_timestamp_mismatch_count",
    ])
    .sort("date")
)

display(daily_rows_enriched)


date,rows,users,items,search,view,click,favorite,to_cart,size_mb,min_timestamp,max_timestamp,date_timestamp_mismatch_count
date,i64,u32,u32,i64,i64,i64,i64,i64,f64,datetime[μs],datetime[μs],i64
2024-03-01,10446059,164133,39956,592151,9280153,230035,10242,333478,212.883232,2024-03-01 00:00:00,2024-03-01 23:59:59,0
2024-03-02,11454573,174892,41102,642899,10185851,254477,9847,361499,232.820042,2024-03-02 00:00:00,2024-03-02 23:59:59,0
2024-03-03,11127161,177518,41059,624602,9899880,245688,10263,346728,227.423747,2024-03-03 00:00:00,2024-03-03 23:59:59,0
2024-03-04,9638934,166176,40273,538250,8589520,212654,9699,288811,198.996922,2024-03-04 00:00:00,2024-03-04 23:59:59,0
2024-03-05,10859997,178864,40878,589928,9708096,238194,10410,313369,222.475039,2024-03-05 00:00:00,2024-03-05 23:59:59,0
…,…,…,…,…,…,…,…,…,…,…,…,…
2024-04-26,7957682,138570,39913,519806,6992884,174672,7159,263161,157.273384,2024-04-26 00:00:00,2024-04-26 23:59:59,0
2024-04-27,9534450,141475,39758,560453,8434003,210528,7692,321774,188.11931,2024-04-27 00:00:00,2024-04-27 23:59:59,0
2024-04-28,8831496,135816,39664,526365,7810240,199960,7446,287485,175.897675,2024-04-28 00:00:00,2024-04-28 23:59:59,0


### User actions overview conclusions

- По partition-level подсчету в логах найдено 561 836 992 событий.
- Все ожидаемые типы событий присутствуют: `view`, `search`, `to_cart`, `click`, `favorite`.
- Данные покрывают 61 день: с 2024-03-01 по 2024-04-30.
- Для каждого дня есть все 5 ожидаемых action-type партиций.
- Daily readiness table содержит строки, пользователей, товары, распределение событий и проверку соответствия `date` и даты из `timestamp`.
- Дневной объем событий заметно различается, поэтому дальнейшие блоки отдельно проверяют пользователей, товары, аномалии и popularity bias.


## 6. Action type analysis


In [23]:
action_type_metadata = pl.DataFrame({
    "action_type": ["search", "view", "click", "favorite", "to_cart"],
    "meaning": [
        "search query",
        "product/card view",
        "product click",
        "add to favorites",
        "add to cart",
    ],
    "baseline_usage": [
        "exclude from item-pairs; keep for future co-search",
        "use for co-visitation as weak signal",
        "use for co-visitation as stronger signal",
        "use for co-visitation as saved-interest signal",
        "use for co-visitation as strongest current signal",
    ],
    "weight": [None, 1.0, 2.0, 2.5, 4.0],
})

print("Action type metadata; final action-type decision table is built after missing-value checks.")
display(action_type_metadata)


Action type metadata; final action-type decision table is built after missing-value checks.


action_type,meaning,baseline_usage,weight
str,str,str,f64
"""search""","""search query""","""exclude from item-pairs; keep …",null
"""view""","""product/card view""","""use for co-visitation as weak …",1.0
"""click""","""product click""","""use for co-visitation as stron…",2.0
"""favorite""","""add to favorites""","""use for co-visitation as saved…",2.5
"""to_cart""","""add to cart""","""use for co-visitation as stron…",4.0


In [24]:
expected_action_type_set = set(action_type_metadata.get_column("action_type").to_list())
observed_action_type_set = set(rows_by_partition.get_column("action_type").unique().to_list())

action_type_guardrail = pl.DataFrame({
    "check": [
        "unexpected_action_types",
        "missing_expected_action_types",
    ],
    "values": [
        sorted(observed_action_type_set - expected_action_type_set),
        sorted(expected_action_type_set - observed_action_type_set),
    ],
    "count": [
        len(observed_action_type_set - expected_action_type_set),
        len(expected_action_type_set - observed_action_type_set),
    ],
})

print("Action type guardrail checks:")
display(action_type_guardrail)

if action_type_guardrail.get_column("count").sum() > 0:
    raise RuntimeError("Unexpected or missing action_type values found in rows_by_partition.")


Action type guardrail checks:


check,values,count
str,list[null],i64
"""unexpected_action_types""",[],0
"""missing_expected_action_types""",[],0


### Action type analysis conclusions

- Для co-visitation baseline используем товарные события: `view`, `click`, `favorite`, `to_cart`.
- `search` не используем напрямую для item-to-item графа, потому что поисковое событие описывает запрос, а не обязательно взаимодействие с конкретным товаром.
- `search` стоит сохранить отдельно для будущего co-search сигнала.
- Стартовые веса событий для baseline: `view = 1.0`, `click = 2.0`, `favorite = 2.5`, `to_cart = 4.0`.
- Основная доля событий приходится на `view`, поэтому в baseline нужна защита от popularity bias.


## 7. User actions missing values analysis


In [25]:
sample_day_nulls = (
    sample_day_lf
    .select([
        pl.len().alias("rows"),

        pl.col("user_id").is_null().sum().alias("user_id_null_count"),
        pl.col("date").is_null().sum().alias("date_null_count"),
        pl.col("timestamp").is_null().sum().alias("timestamp_null_count"),
        pl.col("action_type").is_null().sum().alias("action_type_null_count"),
        pl.col("widget_name").is_null().sum().alias("widget_name_null_count"),
        pl.col("search_query").is_null().sum().alias("search_query_null_count"),
        pl.col("item_id").is_null().sum().alias("item_id_null_count"),

        (
            pl.col("search_query").is_not_null()
            & (pl.col("search_query").str.strip_chars() == "")
        ).sum().alias("search_query_blank_count"),

        (
            pl.col("widget_name").is_not_null()
            & (pl.col("widget_name").str.strip_chars() == "")
        ).sum().alias("widget_name_blank_count"),
    ])
    .with_columns([
        (pl.col("user_id_null_count") / pl.col("rows")).alias("user_id_null_share"),
        (pl.col("timestamp_null_count") / pl.col("rows")).alias("timestamp_null_share"),
        (pl.col("action_type_null_count") / pl.col("rows")).alias("action_type_null_share"),
        (pl.col("item_id_null_count") / pl.col("rows")).alias("item_id_null_share"),
        (pl.col("search_query_null_count") / pl.col("rows")).alias("search_query_null_share"),
    ])
    .collect()
)

print("Sample day:", smallest_day)
display(sample_day_nulls)


Sample day: 2024-04-24


rows,user_id_null_count,date_null_count,timestamp_null_count,action_type_null_count,widget_name_null_count,search_query_null_count,item_id_null_count,search_query_blank_count,widget_name_blank_count,user_id_null_share,timestamp_null_share,action_type_null_share,item_id_null_share,search_query_null_share
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,f64,f64,f64,f64,f64
7431439,0,0,0,0,0,1190,441598,5,0,0.0,0.0,0.0,0.059423,0.00016


In [26]:
sample_day_missing_by_action = (
    sample_day_lf
    .group_by("action_type")
    .agg([
        pl.len().alias("rows"),

        pl.col("user_id").is_null().sum().alias("user_id_null_count"),
        pl.col("timestamp").is_null().sum().alias("timestamp_null_count"),
        pl.col("item_id").is_null().sum().alias("item_id_null_count"),
        pl.col("search_query").is_null().sum().alias("search_query_null_count"),
        pl.col("widget_name").is_null().sum().alias("widget_name_null_count"),

        (
            pl.col("search_query").is_not_null()
            & (pl.col("search_query").str.strip_chars() == "")
        ).sum().alias("search_query_blank_count"),

        (
            pl.col("widget_name").is_not_null()
            & (pl.col("widget_name").str.strip_chars() == "")
        ).sum().alias("widget_name_blank_count"),
    ])
    .with_columns([
        (pl.col("user_id_null_count") / pl.col("rows")).alias("user_id_null_share"),
        (pl.col("timestamp_null_count") / pl.col("rows")).alias("timestamp_null_share"),
        (pl.col("item_id_null_count") / pl.col("rows")).alias("item_id_null_share"),
        (pl.col("search_query_null_count") / pl.col("rows")).alias("search_query_null_share"),
        (pl.col("widget_name_null_count") / pl.col("rows")).alias("widget_name_null_share"),
    ])
    .sort("rows", descending=True)
    .collect()
)

display(sample_day_missing_by_action)


action_type,rows,user_id_null_count,timestamp_null_count,item_id_null_count,search_query_null_count,widget_name_null_count,search_query_blank_count,widget_name_blank_count,user_id_null_share,timestamp_null_share,item_id_null_share,search_query_null_share,widget_name_null_share
str,u32,u32,u32,u32,u32,u32,u32,u32,f64,f64,f64,f64,f64
"""view""",6569481,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0
"""search""",441598,0,0,441598,1190,0,5,0,0.0,0.0,1.0,0.002695,0.0
"""to_cart""",249438,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0
"""click""",163953,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0
"""favorite""",6969,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0


In [27]:
MISSING_BY_PARTITION_PATH = CACHE_DIR / "eda_missing_by_partition.parquet"

if should_load_eda_cache(MISSING_BY_PARTITION_PATH, "missing_by_partition"):
    missing_by_partition = read_eda_cache(MISSING_BY_PARTITION_PATH, "missing_by_partition")
else:
    missing_records = []

    for i, row in enumerate(file_inventory.iter_rows(named=True), start=1):
        partition_missing = (
            pl.scan_parquet(row["path"], hive_partitioning=True)
            .select([
                pl.len().alias("rows"),

                pl.col("user_id").is_null().sum().alias("user_id_null_count"),
                pl.col("date").is_null().sum().alias("date_null_count"),
                pl.col("timestamp").is_null().sum().alias("timestamp_null_count"),
                pl.col("action_type").is_null().sum().alias("action_type_null_count"),
                pl.col("widget_name").is_null().sum().alias("widget_name_null_count"),
                pl.col("search_query").is_null().sum().alias("search_query_null_count"),
                pl.col("item_id").is_null().sum().alias("item_id_null_count"),

                (
                    pl.col("search_query").is_not_null()
                    & (pl.col("search_query").str.strip_chars() == "")
                ).sum().alias("search_query_blank_count"),

                (
                    pl.col("widget_name").is_not_null()
                    & (pl.col("widget_name").str.strip_chars() == "")
                ).sum().alias("widget_name_blank_count"),
            ])
            .collect()
        )

        record = partition_missing.to_dicts()[0]
        record.update({
            "date": row["date"],
            "action_type": row["action_type"],
            "size_mb": row["size_mb"],
            "relative_path": row["relative_path"],
        })

        missing_records.append(record)

        if i % 25 == 0:
            print(f"Processed {i} / {file_inventory.height} files")

    missing_by_partition = pl.DataFrame(missing_records)
    write_eda_cache(missing_by_partition, MISSING_BY_PARTITION_PATH, "missing_by_partition")

display(missing_by_partition.head(10))


Processed 25 / 305 files
Processed 50 / 305 files
Processed 75 / 305 files
Processed 100 / 305 files
Processed 125 / 305 files
Processed 150 / 305 files
Processed 175 / 305 files
Processed 200 / 305 files
Processed 225 / 305 files
Processed 250 / 305 files
Processed 275 / 305 files
Processed 300 / 305 files
Saved missing_by_partition cache: C:\Users\User\Projects\OZON-Similar-products\outputs\interim\eda_missing_by_partition.parquet


rows,user_id_null_count,date_null_count,timestamp_null_count,action_type_null_count,widget_name_null_count,search_query_null_count,item_id_null_count,search_query_blank_count,widget_name_blank_count,date,action_type,size_mb,relative_path
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,date,str,f64,str
230035,0,0,0,0,0,0,0,0,0,2024-03-01,"""click""",4.607059,"""data/raw/user_actions/user_act…"
10242,0,0,0,0,0,0,0,0,0,2024-03-01,"""favorite""",0.259104,"""data/raw/user_actions/user_act…"
592151,0,0,0,0,0,810,592151,25,0,2024-03-01,"""search""",12.670323,"""data/raw/user_actions/user_act…"
333478,0,0,0,0,0,0,0,0,0,2024-03-01,"""to_cart""",5.168841,"""data/raw/user_actions/user_act…"
9280153,0,0,0,0,0,0,0,0,0,2024-03-01,"""view""",190.177905,"""data/raw/user_actions/user_act…"
254477,0,0,0,0,0,0,0,0,0,2024-03-02,"""click""",5.151161,"""data/raw/user_actions/user_act…"
9847,0,0,0,0,0,0,0,1,0,2024-03-02,"""favorite""",0.256618,"""data/raw/user_actions/user_act…"
642899,0,0,0,0,0,1054,642899,18,0,2024-03-02,"""search""",13.676371,"""data/raw/user_actions/user_act…"
361499,0,0,0,0,0,0,0,0,0,2024-03-02,"""to_cart""",5.720272,"""data/raw/user_actions/user_act…"


In [28]:
missing_count_columns = [
    "user_id_null_count",
    "date_null_count",
    "timestamp_null_count",
    "action_type_null_count",
    "widget_name_null_count",
    "search_query_null_count",
    "item_id_null_count",
    "search_query_blank_count",
    "widget_name_blank_count",
]

missing_by_action_type = (
    missing_by_partition
    .group_by("action_type")
    .agg([
        pl.col("rows").sum().alias("rows"),
        *[
            pl.col(column).sum().alias(column)
            for column in missing_count_columns
        ],
    ])
    .with_columns([
        *[
            (pl.col(column) / pl.col("rows")).alias(column.replace("_count", "_share"))
            for column in missing_count_columns
        ],
    ])
    .sort("rows", descending=True)
)

missing_decision_view = (
    missing_by_action_type
    .select([
        "action_type",
        "rows",
        "user_id_null_count",
        "user_id_null_share",
        "timestamp_null_count",
        "timestamp_null_share",
        "item_id_null_count",
        "item_id_null_share",
        "search_query_null_count",
        "search_query_null_share",
        "search_query_blank_count",
        "search_query_blank_share",
    ])
    .sort("rows", descending=True)
)

display(missing_by_action_type)
display(missing_decision_view)


action_type,rows,user_id_null_count,date_null_count,timestamp_null_count,action_type_null_count,widget_name_null_count,search_query_null_count,item_id_null_count,search_query_blank_count,widget_name_blank_count,user_id_null_share,date_null_share,timestamp_null_share,action_type_null_share,widget_name_null_share,search_query_null_share,item_id_null_share,search_query_blank_share,widget_name_blank_share
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""view""",498735607,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""search""",32274194,0,0,0,0,0,84215,32274194,773,0,0.0,0.0,0.0,0.0,0.0,0.002609,1.0,0.000024,0.0
"""to_cart""",18110504,0,0,0,0,0,0,0,2,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.1043e-7,0.0
"""click""",12224896,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""favorite""",491791,0,0,0,0,0,0,0,3,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000006,0.0


action_type,rows,user_id_null_count,user_id_null_share,timestamp_null_count,timestamp_null_share,item_id_null_count,item_id_null_share,search_query_null_count,search_query_null_share,search_query_blank_count,search_query_blank_share
str,i64,i64,f64,i64,f64,i64,f64,i64,f64,i64,f64
"""view""",498735607,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
"""search""",32274194,0,0.0,0,0.0,32274194,1.0,84215,0.002609,773,0.000024
"""to_cart""",18110504,0,0.0,0,0.0,0,0.0,0,0.0,2,1.1043e-7
"""click""",12224896,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
"""favorite""",491791,0,0.0,0,0.0,0,0.0,0,0.0,3,0.000006


In [29]:
action_type_analysis = (
    action_type_rows
    .join(
        missing_by_action_type.select([
            "action_type",
            "item_id_null_share",
            "search_query_null_share",
            "search_query_blank_share",
        ]),
        on="action_type",
        how="left",
    )
    .join(action_type_metadata, on="action_type", how="left")
    .select([
        "action_type",
        "rows",
        "share",
        "days",
        "size_mb",
        "item_id_null_share",
        "search_query_null_share",
        "search_query_blank_share",
        "meaning",
        "baseline_usage",
        "weight",
    ])
    .sort("rows", descending=True)
)

display(action_type_analysis)


action_type,rows,share,days,size_mb,item_id_null_share,search_query_null_share,search_query_blank_share,meaning,baseline_usage,weight
str,i64,f64,u32,f64,f64,f64,f64,str,str,f64
"""view""",498735607,0.887687,61,10026.76433,0.0,0.0,0.0,"""product/card view""","""use for co-visitation as weak …",1.0
"""search""",32274194,0.057444,61,682.219216,1.0,0.002609,0.000024,"""search query""","""exclude from item-pairs; keep …",null
"""to_cart""",18110504,0.032234,61,268.73946,0.0,0.0,1.1043e-7,"""add to cart""","""use for co-visitation as stron…",4.0
"""click""",12224896,0.021759,61,226.028446,0.0,0.0,0.0,"""product click""","""use for co-visitation as stron…",2.0
"""favorite""",491791,0.000875,61,12.612441,0.0,0.0,0.000006,"""add to favorites""","""use for co-visitation as saved…",2.5


In [30]:
missing_preprocessing_decisions = pl.DataFrame({
    "condition": [
        "user_id is null",
        "timestamp is null",
        "action_type is null",
        "action_type = search and item_id is null",
        "action_type in view/click/favorite/to_cart and item_id is null",
        "action_type = search and search_query is null or blank",
        "search_query is null or blank for non-search events",
    ],
    "decision": [
        "drop",
        "drop",
        "drop",
        "keep separately, do not use for item-to-item graph",
        "drop for co-visitation graph",
        "drop from future co-search analysis",
        "ignore for item-to-item baseline",
    ],
    "reason": [
        "нельзя построить историю пользователя",
        "нельзя упорядочить события во времени",
        "нельзя назначить вес события",
        "search описывает запрос, а не товарное взаимодействие",
        "нельзя построить item-pair без item_id",
        "нельзя использовать пустой запрос для co-search",
        "для товарных событий search_query не является обязательным полем",
    ],
})

display(missing_preprocessing_decisions)


condition,decision,reason
str,str,str
"""user_id is null""","""drop""","""нельзя построить историю польз…"
"""timestamp is null""","""drop""","""нельзя упорядочить события во …"
"""action_type is null""","""drop""","""нельзя назначить вес события"""
"""action_type = search and item_…","""keep separately, do not use fo…","""search описывает запрос, а не …"
"""action_type in view/click/favo…","""drop for co-visitation graph""","""нельзя построить item-pair без…"
"""action_type = search and searc…","""drop from future co-search ana…","""нельзя использовать пустой зап…"
"""search_query is null or blank …","""ignore for item-to-item baseli…","""для товарных событий search_qu…"


### Missing values conclusions

- В `user_id` и `timestamp` пропусков нет для всех типов событий.
- В товарных событиях `view`, `click`, `favorite`, `to_cart` пропусков `item_id` нет.
- У `search` `item_id` отсутствует в 100% строк. Это ожидаемо: поисковое событие описывает запрос, а не конкретный товар.
- У `search` есть небольшая доля null и blank значений `search_query`; такие строки нужно исключать из будущего co-search анализа.
- Для item-to-item baseline среди товарных событий `view/click/favorite/to_cart` удалять строки по missing values не требуется. `search`-события не входят в item-to-item graph и должны обрабатываться отдельно.


## 8. Диагностика дубликатов в user_actions

Этот раздел проверяет точные повторяющиеся сырые события по ключу `user_id`, `timestamp`, `action_type`, `item_id`, `search_query`, `widget_name`.

Проверка дубликатов выполняется на двух безопасных для EDA уровнях:
- диагностика точных дубликатов за один sample-day для `sample_day = 2024-04-24`;
- кешированная проверка точных дубликатов по всем parquet-партициям `date x action_type`.

Сквозная проверка точных дубликатов между партициями не выполняется, потому что она требует глобального многостолбцового group_by по всем сырым событиям.

In [31]:
duplicate_event_check_scope = pl.DataFrame({
    "check": [
        "sample_day_exact_duplicate_check",
        "per_partition_exact_duplicate_check",
        "global_cross_partition_duplicate_check",
    ],
    "status": [
        "executed as EDA diagnostic",
        "executed and cached as EDA diagnostic",
        "not executed in EDA Step 4",
    ],
    "reason": [
        "Sample-day check is cheap and catches obvious raw-event duplicate risk on the diagnostic day.",
        "Current layout has one parquet file per date x action_type partition; this catches duplicates inside each partition without a global shuffle.",
        "Exact cross-partition duplicate detection over all raw user_actions would require a global multi-column group_by over 11+ GB.",
    ],
    "decision": [
        "use as diagnostic evidence, not global proof",
        "use as stronger EDA evidence for duplicate risk inside partitions",
        "treat cross-partition duplicate risk as not globally verified; validate in preprocessing or a dedicated out-of-core job if required",
    ],
})

print("Duplicate event check scope:")
display(duplicate_event_check_scope)


Duplicate event check scope:


check,status,reason,decision
str,str,str,str
"""sample_day_exact_duplicate_che…","""executed as EDA diagnostic""","""Sample-day check is cheap and …","""use as diagnostic evidence, no…"
"""per_partition_exact_duplicate_…","""executed and cached as EDA dia…","""Current layout has one parquet…","""use as stronger EDA evidence f…"
"""global_cross_partition_duplica…","""not executed in EDA Step 4""","""Exact cross-partition duplicat…","""treat cross-partition duplicat…"


In [32]:
duplicate_key_columns = [
    "user_id",
    "timestamp",
    "action_type",
    "item_id",
    "search_query",
    "widget_name",
]

sample_day_duplicate_summary = (
    sample_day_lf
    .group_by(duplicate_key_columns)
    .agg(pl.len().alias("duplicate_group_size"))
    .filter(pl.col("duplicate_group_size") > 1)
    .select([
        pl.len().alias("duplicate_groups"),
        pl.col("duplicate_group_size").sum().alias("duplicate_rows_in_groups"),
        (pl.col("duplicate_group_size") - 1).sum().alias("extra_duplicate_rows"),
        pl.col("duplicate_group_size").max().alias("max_duplicate_group_size"),
    ])
    .collect(engine="streaming")
)

sample_day_duplicate_summary = sample_day_duplicate_summary.with_columns([
    pl.lit(smallest_day).alias("sample_day"),
    (pl.col("extra_duplicate_rows") / pl.lit(sample_day_overview.get_column("rows").item())).alias("extra_duplicate_rows_share"),
]).select([
    "sample_day",
    "duplicate_groups",
    "duplicate_rows_in_groups",
    "extra_duplicate_rows",
    "extra_duplicate_rows_share",
    "max_duplicate_group_size",
])

print("Sample-day exact duplicate diagnostic:")
display(sample_day_duplicate_summary)

DUPLICATE_CHECK_BY_PARTITION_PATH = CACHE_DIR / "eda_duplicate_check_by_partition.parquet"

if should_load_eda_cache(DUPLICATE_CHECK_BY_PARTITION_PATH, "duplicate_check_by_partition"):
    duplicate_check_by_partition = read_eda_cache(DUPLICATE_CHECK_BY_PARTITION_PATH, "duplicate_check_by_partition")
else:
    duplicate_records = []

    for i, row in enumerate(file_inventory.iter_rows(named=True), start=1):
        partition_duplicate_summary = (
            pl.scan_parquet(row["path"], hive_partitioning=True)
            .group_by(duplicate_key_columns)
            .agg(pl.len().alias("duplicate_group_size"))
            .filter(pl.col("duplicate_group_size") > 1)
            .select([
                pl.len().alias("duplicate_groups"),
                pl.col("duplicate_group_size").sum().alias("duplicate_rows_in_groups"),
                (pl.col("duplicate_group_size") - 1).sum().alias("extra_duplicate_rows"),
                pl.col("duplicate_group_size").max().alias("max_duplicate_group_size"),
            ])
            .collect(engine="streaming")
        )

        record = partition_duplicate_summary.to_dicts()[0]
        record.update({
            "date": row["date"],
            "action_type": row["action_type"],
            "rows": rows_by_partition.filter(
                (pl.col("date") == row["date"])
                & (pl.col("action_type") == row["action_type"])
            ).get_column("rows").item(),
            "relative_path": row["relative_path"],
        })
        duplicate_records.append(record)

        if i % 25 == 0:
            print(f"Processed {i} / {file_inventory.height} duplicate-check partitions")

    duplicate_check_by_partition = pl.DataFrame(duplicate_records).with_columns([
        (pl.col("extra_duplicate_rows") / pl.col("rows")).alias("extra_duplicate_rows_share"),
    ]).select([
        "date",
        "action_type",
        "rows",
        "duplicate_groups",
        "duplicate_rows_in_groups",
        "extra_duplicate_rows",
        "extra_duplicate_rows_share",
        "max_duplicate_group_size",
        "relative_path",
    ])

    write_eda_cache(duplicate_check_by_partition, DUPLICATE_CHECK_BY_PARTITION_PATH, "duplicate_check_by_partition")

per_partition_duplicate_summary = duplicate_check_by_partition.select([
    pl.len().alias("partitions_checked"),
    pl.col("rows").sum().alias("rows_checked"),
    (pl.col("duplicate_groups") > 0).sum().alias("partitions_with_duplicate_groups"),
    pl.col("duplicate_groups").sum().alias("duplicate_groups"),
    pl.col("duplicate_rows_in_groups").sum().alias("duplicate_rows_in_groups"),
    pl.col("extra_duplicate_rows").sum().alias("extra_duplicate_rows"),
    pl.col("max_duplicate_group_size").max().alias("max_duplicate_group_size"),
]).with_columns([
    (pl.col("extra_duplicate_rows") / pl.col("rows_checked")).alias("extra_duplicate_rows_share"),
])

print("Per-partition exact duplicate diagnostic summary:")
display(per_partition_duplicate_summary)

print("Partitions with duplicate groups, if any:")
display(
    duplicate_check_by_partition
    .filter(pl.col("duplicate_groups") > 0)
    .sort(["extra_duplicate_rows", "duplicate_groups"], descending=[True, True])
    .head(30)
)


Sample-day exact duplicate diagnostic:


sample_day,duplicate_groups,duplicate_rows_in_groups,extra_duplicate_rows,extra_duplicate_rows_share,max_duplicate_group_size
date,u32,u32,u32,f64,u32
2024-04-24,4421,8978,4557,0.000613,12


Processed 25 / 305 duplicate-check partitions
Processed 50 / 305 duplicate-check partitions
Processed 75 / 305 duplicate-check partitions
Processed 100 / 305 duplicate-check partitions
Processed 125 / 305 duplicate-check partitions
Processed 150 / 305 duplicate-check partitions
Processed 175 / 305 duplicate-check partitions
Processed 200 / 305 duplicate-check partitions
Processed 225 / 305 duplicate-check partitions
Processed 250 / 305 duplicate-check partitions
Processed 275 / 305 duplicate-check partitions
Processed 300 / 305 duplicate-check partitions
Saved duplicate_check_by_partition cache: C:\Users\User\Projects\OZON-Similar-products\outputs\interim\eda_duplicate_check_by_partition.parquet
Per-partition exact duplicate diagnostic summary:


partitions_checked,rows_checked,partitions_with_duplicate_groups,duplicate_groups,duplicate_rows_in_groups,extra_duplicate_rows,max_duplicate_group_size,extra_duplicate_rows_share
u32,i64,u32,i64,i64,i64,i64,f64
305,561836992,299,571156,1159547,588391,40,0.001047


Partitions with duplicate groups, if any:


date,action_type,rows,duplicate_groups,duplicate_rows_in_groups,extra_duplicate_rows,extra_duplicate_rows_share,max_duplicate_group_size,relative_path
date,str,i64,i64,i64,i64,f64,i64,str
2024-04-30,"""search""",483214,18135,36364,18229,0.037724,8,"""data/raw/user_actions/user_act…"
2024-04-28,"""search""",526365,16719,33589,16870,0.03205,12,"""data/raw/user_actions/user_act…"
2024-04-29,"""search""",478544,16624,33372,16748,0.034998,14,"""data/raw/user_actions/user_act…"
2024-04-27,"""search""",560453,14165,28437,14272,0.025465,10,"""data/raw/user_actions/user_act…"
2024-03-16,"""view""",8899391,10886,21900,11014,0.001238,4,"""data/raw/user_actions/user_act…"
…,…,…,…,…,…,…,…,…
2024-03-21,"""view""",8095408,7397,14869,7472,0.000923,4,"""data/raw/user_actions/user_act…"
2024-03-29,"""search""",542643,7302,14761,7459,0.013746,10,"""data/raw/user_actions/user_act…"
2024-03-24,"""search""",637481,7095,14430,7335,0.011506,27,"""data/raw/user_actions/user_act…"


### Duplicate diagnostics conclusions

- On `sample_day = 2024-04-24`, exact raw-event duplicate diagnostics found 4,421 duplicate groups and 4,557 extra duplicate rows, about 0.0613% of sample-day rows.
- The cached per-partition check across all 305 parquet partitions found duplicate groups in 299 partitions.
- In total, the per-partition check found 571,156 duplicate groups and 588,391 extra duplicate rows, about 0.1047% of all `user_actions` events.
- The maximum duplicate group size inside one partition is 40 events.
- The duplicate share is small, but duplicates are confirmed. Before building the co-visitation baseline, preprocessing should decide whether to deduplicate raw events because repeats can slightly inflate popularity, event weights, and pair counts.
- Cross-partition exact duplicate detection remains outside EDA Step 4 and should be handled by a dedicated out-of-core data-quality job if needed.


## 9. User analysis


In [33]:
sample_day_user_activity = (
    sample_day_lf
    .group_by("user_id")
    .agg([
        pl.len().alias("events"),
        pl.col("item_id").drop_nulls().n_unique().alias("unique_items"),
        pl.col("timestamp").min().alias("first_timestamp"),
        pl.col("timestamp").max().alias("last_timestamp"),
    ])
    .sort("events", descending=True)
    .collect()
)

display(sample_day_user_activity.head(20))


user_id,events,unique_items,first_timestamp,last_timestamp
i32,u32,u32,datetime[ns],datetime[ns]
6195822,259919,27922,2024-04-24 00:00:00,2024-04-24 23:57:19
4809133,3937,1973,2024-04-24 05:21:56,2024-04-24 13:14:33
9421240,2947,46,2024-04-24 07:14:27,2024-04-24 11:52:13
10041676,2115,1093,2024-04-24 09:26:08,2024-04-24 18:22:58
8496361,1856,1410,2024-04-24 00:51:55,2024-04-24 13:48:27
…,…,…,…,…
5940590,1281,884,2024-04-24 12:39:23,2024-04-24 18:40:29
8396965,1266,913,2024-04-24 08:03:21,2024-04-24 11:49:49
5176619,1233,847,2024-04-24 11:12:04,2024-04-24 20:59:27


In [34]:
sample_day_user_activity_summary = sample_day_user_activity.select([
    pl.len().alias("users"),
    pl.col("events").sum().alias("events"),
    pl.col("events").mean().alias("mean_events_per_user"),
    pl.col("events").median().alias("median_events_per_user"),
    pl.col("events").quantile(0.90).alias("p90_events_per_user"),
    pl.col("events").quantile(0.95).alias("p95_events_per_user"),
    pl.col("events").quantile(0.99).alias("p99_events_per_user"),
    pl.col("events").max().alias("max_events_per_user"),
    pl.col("unique_items").median().alias("median_unique_items_per_user"),
    pl.col("unique_items").quantile(0.99).alias("p99_unique_items_per_user"),
    pl.col("unique_items").max().alias("max_unique_items_per_user"),
])

sample_day_user_thresholds = sample_day_user_activity.select([
    (pl.col("events") >= 100).sum().alias("users_with_100_plus_events"),
    (pl.col("events") >= 500).sum().alias("users_with_500_plus_events"),
    (pl.col("events") >= 1000).sum().alias("users_with_1000_plus_events"),
    (pl.col("unique_items") >= 100).sum().alias("users_with_100_plus_unique_items"),
    (pl.col("unique_items") >= 500).sum().alias("users_with_500_plus_unique_items"),
])

display(sample_day_user_activity_summary)
display(sample_day_user_thresholds)


users,events,mean_events_per_user,median_events_per_user,p90_events_per_user,p95_events_per_user,p99_events_per_user,max_events_per_user,median_unique_items_per_user,p99_unique_items_per_user,max_unique_items_per_user
u32,u32,f64,f64,f64,f64,f64,u32,f64,f64,u32
122779,7431439,60.526955,25.0,154.0,227.0,425.0,259919,19.0,326.0,27922


users_with_100_plus_events,users_with_500_plus_events,users_with_1000_plus_events,users_with_100_plus_unique_items,users_with_500_plus_unique_items
u32,u32,u32,u32,u32
21841,726,42,15704,259


In [35]:
top_active_user_ids = (
    sample_day_user_activity
    .head(20)
    .get_column("user_id")
    .to_list()
)

top_active_users_action_distribution = (
    sample_day_lf
    .filter(pl.col("user_id").is_in(top_active_user_ids))
    .group_by(["user_id", "action_type"])
    .agg(pl.len().alias("events"))
    .sort(["user_id", "events"], descending=[False, True])
    .collect()
)

display(top_active_users_action_distribution)


user_id,action_type,events
i32,str,u32
28699,"""view""",1150
28699,"""search""",42
28699,"""click""",13
28699,"""to_cart""",11
2597773,"""view""",1501
…,…,…
10041676,"""to_cart""",38
10207636,"""view""",1728
10207636,"""to_cart""",51


In [36]:
display(user_daily_activity_summary)

display(
    user_daily_activity_summary
    .sort("max_events_per_user", descending=True)
    .head(10)
)


date,users,events,mean_events_per_user,median_events_per_user,p90_events_per_user,p95_events_per_user,p99_events_per_user,max_events_per_user,median_unique_items_per_user,p99_unique_items_per_user,max_unique_items_per_user
date,u32,u32,f64,f64,f64,f64,f64,u32,f64,f64,u32
2024-03-01,164133,10446059,63.643868,25.0,162.0,241.0,465.0,438686,18.0,339.0,30931
2024-03-02,174892,11454573,65.495123,25.0,167.0,250.0,475.0,508144,18.0,347.0,30736
2024-03-03,177518,11127161,62.681875,24.0,160.0,240.0,453.0,530910,17.0,327.0,30715
2024-03-04,166176,9638934,58.004369,22.0,146.0,220.0,428.0,496340,15.0,309.0,30597
2024-03-05,178864,10859997,60.716505,23.0,154.0,232.0,448.0,515495,16.0,321.0,31022
…,…,…,…,…,…,…,…,…,…,…,…
2024-04-26,138570,7957682,57.427163,22.0,149.0,224.0,429.0,250999,16.0,328.0,28168
2024-04-27,141475,9534450,67.393179,28.0,172.0,253.0,475.0,304292,21.0,366.0,29070
2024-04-28,135816,8831496,65.025446,26.0,166.0,246.0,463.0,332085,20.0,353.0,29726


date,users,events,mean_events_per_user,median_events_per_user,p90_events_per_user,p95_events_per_user,p99_events_per_user,max_events_per_user,median_unique_items_per_user,p99_unique_items_per_user,max_unique_items_per_user
date,u32,u32,f64,f64,f64,f64,f64,u32,f64,f64,u32
2024-03-03,177518,11127161,62.681875,24.0,160.0,240.0,453.0,530910,17.0,327.0,30715
2024-03-05,178864,10859997,60.716505,23.0,154.0,232.0,448.0,515495,16.0,321.0,31022
2024-03-02,174892,11454573,65.495123,25.0,167.0,250.0,475.0,508144,18.0,347.0,30736
2024-03-10,162282,10778894,66.420761,26.0,169.0,250.0,465.0,504732,19.0,338.0,30124
2024-03-04,166176,9638934,58.004369,22.0,146.0,220.0,428.0,496340,15.0,309.0,30597
2024-03-11,150595,9316993,61.867877,25.0,156.0,232.0,438.0,478955,17.0,316.0,30275
2024-03-09,153939,9994673,64.926192,26.0,164.0,244.0,462.0,474600,18.0,333.0,29736
2024-03-07,179867,12215617,67.91472,26.0,172.0,259.0,508.0,470068,19.0,363.0,30557
2024-03-08,169096,10946358,64.734577,25.0,163.0,246.0,479.0,464469,18.0,337.0,29691


In [37]:
user_daily_outlier_view = (
    user_daily_activity_summary
    .with_columns([
        (pl.col("max_events_per_user") / pl.col("p99_events_per_user")).alias("max_to_p99_events_ratio"),
        (pl.col("max_unique_items_per_user") / pl.col("p99_unique_items_per_user")).alias("max_to_p99_unique_items_ratio"),
    ])
    .select([
        "date",
        "users",
        "events",
        "p99_events_per_user",
        "max_events_per_user",
        "max_to_p99_events_ratio",
        "p99_unique_items_per_user",
        "max_unique_items_per_user",
        "max_to_p99_unique_items_ratio",
    ])
    .sort("max_to_p99_events_ratio", descending=True)
)

display(user_daily_outlier_view.head(15))


date,users,events,p99_events_per_user,max_events_per_user,max_to_p99_events_ratio,p99_unique_items_per_user,max_unique_items_per_user,max_to_p99_unique_items_ratio
date,u32,u32,f64,u32,f64,f64,u32,f64
2024-03-03,177518,11127161,453.0,530910,1171.986755,327.0,30715,93.929664
2024-03-04,166176,9638934,428.0,496340,1159.672897,309.0,30597,99.019417
2024-03-05,178864,10859997,448.0,515495,1150.658482,321.0,31022,96.641745
2024-03-11,150595,9316993,438.0,478955,1093.504566,316.0,30275,95.806962
2024-03-10,162282,10778894,465.0,504732,1085.445161,338.0,30124,89.12426
…,…,…,…,…,…,…,…,…
2024-03-07,179867,12215617,508.0,470068,925.330709,363.0,30557,84.179063
2024-03-17,152940,10248518,468.0,420466,898.431624,331.0,29424,88.89426
2024-03-30,149908,9614152,463.0,406276,877.485961,353.0,30691,86.943343


In [38]:
TOP_USER_BY_DAY_PATH = CACHE_DIR / "eda_top_user_by_day.parquet"

if should_load_eda_cache(TOP_USER_BY_DAY_PATH, "top_user_by_day"):
    top_user_by_day = read_eda_cache(TOP_USER_BY_DAY_PATH, "top_user_by_day")
else:
    top_user_records = []

    for i, day in enumerate(daily_rows.get_column("date").to_list(), start=1):
        day_files = (
            file_inventory
            .filter(pl.col("date") == day)
            .get_column("path")
            .to_list()
        )

        day_lf = pl.scan_parquet(day_files, hive_partitioning=True)

        top_user_day = (
            day_lf
            .group_by("user_id")
            .agg([
                pl.len().alias("events"),
                pl.col("item_id").drop_nulls().n_unique().alias("unique_items"),
                pl.col("timestamp").min().alias("first_timestamp"),
                pl.col("timestamp").max().alias("last_timestamp"),
            ])
            .sort("events", descending=True)
            .head(1)
            .with_columns(pl.lit(day).alias("date"))
            .collect()
        )

        top_user_records.append(top_user_day)

        print(f"Processed {i} / {daily_rows.height} days: {day}")

    top_user_by_day = pl.concat(top_user_records).select([
        "date",
        "user_id",
        "events",
        "unique_items",
        "first_timestamp",
        "last_timestamp",
    ])

    write_eda_cache(top_user_by_day, TOP_USER_BY_DAY_PATH, "top_user_by_day")

display(top_user_by_day.sort("events", descending=True).head(20))


Processed 1 / 61 days: 2024-03-01
Processed 2 / 61 days: 2024-03-02
Processed 3 / 61 days: 2024-03-03
Processed 4 / 61 days: 2024-03-04
Processed 5 / 61 days: 2024-03-05
Processed 6 / 61 days: 2024-03-06
Processed 7 / 61 days: 2024-03-07
Processed 8 / 61 days: 2024-03-08
Processed 9 / 61 days: 2024-03-09
Processed 10 / 61 days: 2024-03-10
Processed 11 / 61 days: 2024-03-11
Processed 12 / 61 days: 2024-03-12
Processed 13 / 61 days: 2024-03-13
Processed 14 / 61 days: 2024-03-14
Processed 15 / 61 days: 2024-03-15
Processed 16 / 61 days: 2024-03-16
Processed 17 / 61 days: 2024-03-17
Processed 18 / 61 days: 2024-03-18
Processed 19 / 61 days: 2024-03-19
Processed 20 / 61 days: 2024-03-20
Processed 21 / 61 days: 2024-03-21
Processed 22 / 61 days: 2024-03-22
Processed 23 / 61 days: 2024-03-23
Processed 24 / 61 days: 2024-03-24
Processed 25 / 61 days: 2024-03-25
Processed 26 / 61 days: 2024-03-26
Processed 27 / 61 days: 2024-03-27
Processed 28 / 61 days: 2024-03-28
Processed 29 / 61 days: 2024-

date,user_id,events,unique_items,first_timestamp,last_timestamp
date,i32,u32,u32,datetime[ns],datetime[ns]
2024-03-03,6195822,530910,30715,2024-03-03 00:00:01,2024-03-03 23:59:59
2024-03-05,6195822,515495,31022,2024-03-05 00:00:02,2024-03-05 23:59:59
2024-03-02,6195822,508144,30736,2024-03-02 00:00:01,2024-03-02 23:59:59
2024-03-10,6195822,504732,30124,2024-03-10 00:00:03,2024-03-10 23:59:59
2024-03-04,6195822,496340,30597,2024-03-04 00:00:00,2024-03-04 23:59:58
…,…,…,…,…,…
2024-03-22,6195822,395583,29642,2024-03-22 00:00:00,2024-03-22 23:59:59
2024-03-12,6195822,385759,29075,2024-03-12 00:00:13,2024-03-12 23:59:58
2024-03-14,6195822,384490,29968,2024-03-14 00:00:36,2024-03-14 23:59:53


In [39]:
top_user_frequency = (
    top_user_by_day
    .group_by("user_id")
    .agg([
        pl.len().alias("days_as_top_user"),
        pl.col("events").sum().alias("total_events_as_top_user"),
        pl.col("events").max().alias("max_daily_events"),
        pl.col("unique_items").max().alias("max_daily_unique_items"),
        pl.col("date").min().alias("first_top_day"),
        pl.col("date").max().alias("last_top_day"),
    ])
    .sort(["days_as_top_user", "total_events_as_top_user"], descending=[True, True])
)

display(top_user_frequency)


user_id,days_as_top_user,total_events_as_top_user,max_daily_events,max_daily_unique_items,first_top_day,last_top_day
i32,u32,u32,u32,u32,date,date
6195822,61,21566285,530910,31023,2024-03-01,2024-04-30


In [40]:
user_preprocessing_decisions = pl.DataFrame({
    "condition": [
        "user_id is null",
        "timestamp is null",
        "events per user per day is extremely high",
        "unique items per user per day is extremely high",
        "session contains too many unique items",
    ],
    "decision": [
        "drop",
        "drop",
        "cap or exclude user-day during preprocessing",
        "cap or exclude user-day during preprocessing",
        "truncate or split session",
    ],
    "reason": [
        "нельзя построить историю пользователя",
        "нельзя упорядочить события во времени",
        "аномальный пользователь может создать много случайных item-pairs",
        "слишком широкий набор товаров ухудшает качество co-visitation graph",
        "слишком длинные сессии создают шумные пары товаров",
    ],
})

display(user_preprocessing_decisions)


condition,decision,reason
str,str,str
"""user_id is null""","""drop""","""нельзя построить историю польз…"
"""timestamp is null""","""drop""","""нельзя упорядочить события во …"
"""events per user per day is ext…","""cap or exclude user-day during…","""аномальный пользователь может …"
"""unique items per user per day …","""cap or exclude user-day during…","""слишком широкий набор товаров …"
"""session contains too many uniq…","""truncate or split session""","""слишком длинные сессии создают…"


### User analysis conclusions

- Типичный пользователь делает существенно меньше событий, чем top active users: медианы и p99 по дням сильно ниже максимальных значений.
- По всем 61 дням один и тот же `user_id = 6195822` является самым активным пользователем дня.
- Такой паттерн выглядит как технический или аномальный трафик, а не как обычное пользовательское поведение.
- На preprocessing-этапе нужно ограничивать вклад слишком активных `user-day`, иначе один пользователь может создать большое число шумных item-pairs.
- На EDA-этапе пользователей физически не удаляем, но фиксируем обязательное preprocessing-решение: ограничить или исключить аномальные `user-day` перед построением co-visitation graph.


## 10. Item analysis

Важно: глобальный точный `unique_users_per_item` по всем 11 GB `user_actions` здесь не считается, чтобы не создавать тяжёлый глобальный cardinality scan. Вместо этого используется sample-day проверка `unique_users_per_item` по самому маленькому дню. Глобальная точная версия не входит в EDA Step 4 и требует отдельного out-of-core расчета.


In [41]:
ITEM_ACTION_COUNTS_PATH = CACHE_DIR / "eda_item_action_counts.parquet"

if should_load_eda_cache(ITEM_ACTION_COUNTS_PATH, "item_action_counts_long"):
    item_action_counts_long = read_eda_cache(ITEM_ACTION_COUNTS_PATH, "item_action_counts_long")
else:
    item_action_counts_long = (
        item_action_counts_by_partition
        .group_by(["item_id", "action_type"])
        .agg(pl.col("events").sum().alias("events"))
        .sort(["item_id", "action_type"])
    )

    write_eda_cache(item_action_counts_long, ITEM_ACTION_COUNTS_PATH, "item_action_counts_long")

display(item_action_counts_long.head(20))


Saved item_action_counts_long cache: C:\Users\User\Projects\OZON-Similar-products\outputs\interim\eda_item_action_counts.parquet


item_id,action_type,events
i32,str,u32
1,"""view""",1
2,"""view""",1
3,"""click""",98
3,"""favorite""",3
3,"""to_cart""",225
…,…,…
10,"""view""",42059
11,"""click""",19
11,"""favorite""",2


In [42]:
item_popularity = (
    item_action_counts_long
    .pivot(
        index="item_id",
        on="action_type",
        values="events",
        aggregate_function="sum",
    )
    .fill_null(0)
)

for column in item_action_types:
    if column not in item_popularity.columns:
        item_popularity = item_popularity.with_columns(
            pl.lit(0).alias(column)
        )

item_popularity = (
    item_popularity
    .with_columns([
        pl.col("view").cast(pl.Int64),
        pl.col("click").cast(pl.Int64),
        pl.col("favorite").cast(pl.Int64),
        pl.col("to_cart").cast(pl.Int64),
    ])
    .with_columns([
        (
            pl.col("view")
            + pl.col("click")
            + pl.col("favorite")
            + pl.col("to_cart")
        ).alias("total_events"),

        (
            pl.col("view") * 1.0
            + pl.col("click") * 2.0
            + pl.col("favorite") * 2.5
            + pl.col("to_cart") * 4.0
        ).alias("weighted_events"),
    ])
    .sort("weighted_events", descending=True)
)

display(item_popularity.head(20))


item_id,view,click,favorite,to_cart,total_events,weighted_events
i32,i64,i64,i64,i64,i64,f64
60173,381541,7962,251,94435,484189,775832.5
114040,445768,10564,269,55485,512086,689508.5
214829,350096,4555,165,48646,403462,554202.5
164779,270422,8204,210,59399,338235,524951.0
140814,320381,3997,129,36411,360918,474341.5
…,…,…,…,…,…,…
90632,246819,8766,112,19250,274947,341631.0
209989,173889,6399,121,36395,216804,332569.5
41141,166039,7324,67,37551,210981,331058.5


In [43]:
SAMPLE_DAY_ITEM_USERS_PATH = CACHE_DIR / f"eda_sample_day_item_users_{smallest_day}.parquet"

if should_load_eda_cache(SAMPLE_DAY_ITEM_USERS_PATH, "sample_day_item_users"):
    sample_day_item_users = read_eda_cache(SAMPLE_DAY_ITEM_USERS_PATH, "sample_day_item_users")
else:
    sample_day_item_users = (
        sample_day_lf
        .filter(pl.col("action_type").is_in(item_action_types))
        .filter(pl.col("item_id").is_not_null())
        .group_by("item_id")
        .agg([
            pl.len().alias("events"),
            pl.col("user_id").n_unique().alias("unique_users"),
        ])
        .sort("events", descending=True)
        .collect(engine="streaming")
    )

    write_eda_cache(sample_day_item_users, SAMPLE_DAY_ITEM_USERS_PATH, "sample_day_item_users")

sample_day_item_users_summary = sample_day_item_users.select([
    pl.len().alias("items_with_behavior_sample_day"),
    pl.col("events").mean().alias("mean_events_per_item_sample_day"),
    pl.col("events").median().alias("median_events_per_item_sample_day"),
    pl.col("events").quantile(0.99).alias("p99_events_per_item_sample_day"),
    pl.col("events").max().alias("max_events_per_item_sample_day"),
    pl.col("unique_users").mean().alias("mean_unique_users_per_item_sample_day"),
    pl.col("unique_users").median().alias("median_unique_users_per_item_sample_day"),
    pl.col("unique_users").quantile(0.99).alias("p99_unique_users_per_item_sample_day"),
    pl.col("unique_users").max().alias("max_unique_users_per_item_sample_day"),
])

print("Sample day:", smallest_day)
display(sample_day_item_users_summary)
display(sample_day_item_users.head(20))


Saved sample_day_item_users cache: C:\Users\User\Projects\OZON-Similar-products\outputs\interim\eda_sample_day_item_users_2024-04-24.parquet
Sample day: 2024-04-24


items_with_behavior_sample_day,mean_events_per_item_sample_day,median_events_per_item_sample_day,p99_events_per_item_sample_day,max_events_per_item_sample_day,mean_unique_users_per_item_sample_day,median_unique_users_per_item_sample_day,p99_unique_users_per_item_sample_day,max_unique_users_per_item_sample_day
u32,f64,f64,f64,u32,f64,f64,f64,u32
38988,179.281856,37.0,1672.0,6763,140.036909,26.0,1309.0,4762


item_id,events,unique_users
i32,u32,u32
114040,6763,4762
60173,6071,4080
214829,5633,4059
133075,5391,4248
164779,5200,3536
…,…,…
216624,3797,2975
21399,3737,2633
33283,3732,2566


In [44]:
catalog_items = products.select("item_id").unique()
behavior_items = item_popularity.select("item_id").unique()

items_in_catalog_with_events = catalog_items.join(
    behavior_items,
    on="item_id",
    how="inner",
)

items_in_catalog_without_events = catalog_items.join(
    behavior_items,
    on="item_id",
    how="anti",
)

items_in_logs_not_in_catalog = behavior_items.join(
    catalog_items,
    on="item_id",
    how="anti",
)

catalog_behavior_coverage = pl.DataFrame({
    "metric": [
        "catalog_items",
        "catalog_items_with_events",
        "catalog_items_without_events",
        "behavior_items",
        "behavior_items_in_catalog",
        "behavior_items_not_in_catalog",
    ],
    "value": [
        catalog_items.height,
        items_in_catalog_with_events.height,
        items_in_catalog_without_events.height,
        behavior_items.height,
        items_in_catalog_with_events.height,
        items_in_logs_not_in_catalog.height,
    ],
    "denominator": [
        "catalog_items",
        "catalog_items",
        "catalog_items",
        "behavior_items",
        "behavior_items",
        "behavior_items",
    ],
    "denominator_value": [
        catalog_items.height,
        catalog_items.height,
        catalog_items.height,
        behavior_items.height,
        behavior_items.height,
        behavior_items.height,
    ],
}).with_columns(
    (pl.col("value") / pl.col("denominator_value")).alias("share")
)

display(catalog_behavior_coverage)


metric,value,denominator,denominator_value,share
str,i64,str,i64,f64
"""catalog_items""",130035,"""catalog_items""",130035,1.0
"""catalog_items_with_events""",72793,"""catalog_items""",130035,0.559795
"""catalog_items_without_events""",57242,"""catalog_items""",130035,0.440205
"""behavior_items""",161218,"""behavior_items""",161218,1.0
"""behavior_items_in_catalog""",72793,"""behavior_items""",161218,0.451519
"""behavior_items_not_in_catalog""",88425,"""behavior_items""",161218,0.548481


In [45]:
catalog_item_ids = products.select("item_id").unique()

item_popularity_catalog = (
    item_popularity
    .join(catalog_item_ids, on="item_id", how="inner")
    .sort("weighted_events", descending=True)
)

item_popularity_not_in_catalog = (
    item_popularity
    .join(catalog_item_ids, on="item_id", how="anti")
    .sort("weighted_events", descending=True)
)

item_catalog_split_summary = pl.DataFrame({
    "metric": [
        "behavior_items_total",
        "behavior_items_in_catalog",
        "behavior_items_not_in_catalog",
        "item_events_total",
        "item_events_in_catalog",
        "item_events_not_in_catalog",
    ],
    "value": [
        item_popularity.height,
        item_popularity_catalog.height,
        item_popularity_not_in_catalog.height,
        item_popularity["total_events"].sum(),
        item_popularity_catalog["total_events"].sum(),
        item_popularity_not_in_catalog["total_events"].sum(),
    ],
})

item_catalog_split_summary = item_catalog_split_summary.with_columns(
    (
        pl.col("value")
        / pl.when(pl.col("metric").str.contains("items"))
            .then(pl.lit(item_popularity.height))
            .otherwise(pl.lit(item_popularity["total_events"].sum()))
    ).alias("share")
)

display(item_catalog_split_summary)

print("Top behavior items not found in product catalog:")
display(item_popularity_not_in_catalog.head(30))


metric,value,share
str,i64,f64
"""behavior_items_total""",161218,1.0
"""behavior_items_in_catalog""",72793,0.451519
"""behavior_items_not_in_catalog""",88425,0.548481
"""item_events_total""",529562798,1.0
"""item_events_in_catalog""",526179865,0.993612
"""item_events_not_in_catalog""",3382933,0.006388


Top behavior items not found in product catalog:


item_id,view,click,favorite,to_cart,total_events,weighted_events
i32,i64,i64,i64,i64,i64,f64
96682,158293,3418,108,7553,169372,195611.0
58130,116307,2086,44,9684,128121,159325.0
41390,98372,2764,39,4895,106070,123577.5
36504,104784,969,16,3092,108861,119130.0
133365,35672,2214,43,14703,52632,99019.5
…,…,…,…,…,…,…
36284,38251,1627,91,689,40658,44488.5
180906,22406,1663,32,4273,28374,42904.0
12764,34236,994,133,1401,36764,42160.5


In [46]:
item_popularity_summary = item_popularity.select([
    pl.len().alias("items_with_behavior"),
    pl.col("total_events").sum().alias("total_item_events"),

    pl.col("total_events").mean().alias("mean_events_per_item"),
    pl.col("total_events").median().alias("median_events_per_item"),
    pl.col("total_events").quantile(0.90).alias("p90_events_per_item"),
    pl.col("total_events").quantile(0.95).alias("p95_events_per_item"),
    pl.col("total_events").quantile(0.99).alias("p99_events_per_item"),
    pl.col("total_events").max().alias("max_events_per_item"),

    pl.col("weighted_events").mean().alias("mean_weighted_events_per_item"),
    pl.col("weighted_events").median().alias("median_weighted_events_per_item"),
    pl.col("weighted_events").quantile(0.99).alias("p99_weighted_events_per_item"),
    pl.col("weighted_events").max().alias("max_weighted_events_per_item"),
])

display(item_popularity_summary)


items_with_behavior,total_item_events,mean_events_per_item,median_events_per_item,p90_events_per_item,p95_events_per_item,p99_events_per_item,max_events_per_item,mean_weighted_events_per_item,median_weighted_events_per_item,p99_weighted_events_per_item,max_weighted_events_per_item
u32,i64,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64
161218,529562798,3284.762235,2.0,5557.0,19345.0,67682.0,512086,3702.172788,2.0,77359.0,775832.5


In [47]:
top_items_by_weighted_events = (
    item_popularity
    .join(products, on="item_id", how="left")
    .select([
        "item_id",
        "name",
        "brand",
        "type",
        "category_id",
        "category_name",
        "view",
        "click",
        "favorite",
        "to_cart",
        "total_events",
        "weighted_events",
    ])
    .sort("weighted_events", descending=True)
    .head(30)
)

display(top_items_by_weighted_events)


item_id,name,brand,type,category_id,category_name,view,click,favorite,to_cart,total_events,weighted_events
i32,str,str,str,i32,str,i64,i64,i64,i64,i64,f64
60173,"""Батон ""Нарезной"" 400 г, Коломе…","""Коломенский""","""Хлеб""",848,"""Белый хлеб""",381541,7962,251,94435,484189,775832.5
114040,"""Молоко питьевое ультрапастериз…","""Село Зеленое""","""Молоко""",814,"""Стерилизованное молоко""",445768,10564,269,55485,512086,689508.5
214829,"""Молоко пастеризованное 2,5% 93…","""Простоквашино""","""Молоко""",248,"""Пастеризованное молоко""",350096,4555,165,48646,403462,554202.5
164779,"""Яйца куриные СО, 10 шт, Волжан…","""Волжанин""","""Яйца""",838,"""Яйца""",270422,8204,210,59399,338235,524951.0
140814,"""Молоко пастеризованное 3,2% 14…","""Простоквашино""","""Молоко""",248,"""Пастеризованное молоко""",320381,3997,129,36411,360918,474341.5
…,…,…,…,…,…,…,…,…,…,…,…
158022,"""Яйца куриные 10 шт, Ozon fresh…","""Ozon fresh""","""Яйца""",838,"""Яйца""",234732,2790,81,17363,254966,309966.5
201318,"""Хлеб Ржаной край зерновой, в н…","""Ржаной край""","""Хлеб""",847,"""Темный хлеб""",239762,3272,82,15791,258907,309675.0
65777,"""Томаты Фламенко, сливовидные, …","""Фламенко""","""Овощи""",751,"""Помидоры""",190236,4525,125,26940,221826,307358.5


In [48]:
item_popularity_sorted = item_popularity.sort("total_events", descending=True)

total_item_events = item_popularity_sorted.get_column("total_events").sum()

concentration_records = []

for top_k in [10, 50, 100, 500, 1000, 5000, 10000]:
    top_k_events = (
        item_popularity_sorted
        .head(top_k)
        .get_column("total_events")
        .sum()
    )

    concentration_records.append({
        "top_k_items": top_k,
        "events_in_top_k": top_k_events,
        "share_of_all_item_events": top_k_events / total_item_events,
    })

popularity_concentration = pl.DataFrame(concentration_records)

display(popularity_concentration)


top_k_items,events_in_top_k,share_of_all_item_events
i64,i64,f64
10,3658736,0.006909
50,13234583,0.024992
100,23181425,0.043775
500,77848494,0.147005
1000,126320905,0.238538
5000,329899915,0.622967
10000,439673694,0.830258


In [49]:
long_tail_summary = item_popularity.select([
    pl.len().alias("items_with_behavior"),

    (pl.col("total_events") == 1).sum().alias("items_with_1_event"),
    (pl.col("total_events") <= 5).sum().alias("items_with_5_or_less_events"),
    (pl.col("total_events") <= 10).sum().alias("items_with_10_or_less_events"),
    (pl.col("total_events") <= 100).sum().alias("items_with_100_or_less_events"),

    (pl.col("click") == 0).sum().alias("items_without_clicks"),
    (pl.col("to_cart") == 0).sum().alias("items_without_to_cart"),
    ((pl.col("click") == 0) & (pl.col("to_cart") == 0)).sum().alias("items_without_clicks_and_to_cart"),
])

long_tail_summary = long_tail_summary.with_columns([
    (pl.col("items_with_1_event") / pl.col("items_with_behavior")).alias("items_with_1_event_share"),
    (pl.col("items_with_5_or_less_events") / pl.col("items_with_behavior")).alias("items_with_5_or_less_events_share"),
    (pl.col("items_with_10_or_less_events") / pl.col("items_with_behavior")).alias("items_with_10_or_less_events_share"),
    (pl.col("items_without_clicks_and_to_cart") / pl.col("items_with_behavior")).alias("items_without_clicks_and_to_cart_share"),
])

display(long_tail_summary)


items_with_behavior,items_with_1_event,items_with_5_or_less_events,items_with_10_or_less_events,items_with_100_or_less_events,items_without_clicks,items_without_to_cart,items_without_clicks_and_to_cart,items_with_1_event_share,items_with_5_or_less_events_share,items_with_10_or_less_events_share,items_without_clicks_and_to_cart_share
u32,u32,u32,u32,u32,u32,u32,u32,f64,f64,f64,f64
161218,79139,96209,98794,118697,100469,119567,97124,0.490882,0.596763,0.612798,0.602439


In [50]:
products_without_behavior = (
    products
    .join(
        item_popularity.select("item_id"),
        on="item_id",
        how="anti",
    )
)

print("Products without behavioral events:", products_without_behavior.height)

display(products_without_behavior.head(30))


Products without behavioral events: 57242


item_id,name,brand,type,category_id,category_name
i32,str,str,str,i32,str
74369,"""Цинк Селен + Витамин Е 100 таб…","""Miopharm""","""Витаминный комплекс""",761,"""Витамин C"""
220368,"""Витамин C липосомальный жидкий…","""SmartLife""","""Витамины""",761,"""Витамин C"""
180477,"""Морской Коллаген и Гиалуронова…","""Maxler""","""Витамины""",761,"""Витамин C"""
152204,"""Витамин С с биофлавоноидами и …","""SmartLife""","""Витамины""",761,"""Витамин C"""
100396,"""Витамины кожа, волосы, ногти Э…","""Эвалар""","""Витаминный комплекс""",761,"""Витамин C"""
…,…,…,…,…,…
27716,"""TRESemme сухой шампунь day 2 д…","""Tresemme""","""Шампунь сухой""",502,"""Сухие шампуни"""
208606,"""Сухой шампунь для объема волос…","""Прелесть""","""Шампунь сухой""",502,"""Сухие шампуни"""
110847,"""The Act labs, Ламеллярный крем…","""The Act""","""Крем для ухода за кожей""",84,"""Кремы для рук"""


In [51]:
item_preprocessing_decisions = pl.DataFrame({
    "condition": [
        "item_id is not in product catalog and item is considered as recommendation candidate",
        "item_id is not in product catalog and item appears in behavioral evidence",
        "catalog item has no behavioral events",
        "item has very few behavioral events",
        "item is extremely popular",
        "pair contains very popular items",
        "item has too few unique users",
    ],
    "decision": [
        "exclude from recommendation candidates for MVP",
        "needs manual preprocessing decision: keep as context evidence or exclude from graph",
        "use fallback instead of co-visitation",
        "keep as candidate, but rely on fallback if graph evidence is weak",
        "keep, but normalize popularity contribution",
        "apply popularity normalization and min_pair_count / min_unique_users",
        "keep only with fallback or require min_unique_users for graph evidence",
    ],
    "reason": [
        "catalog metadata is required to display and interpret recommendations",
        "removing behavioral rows can change co-visitation evidence, so this is not an automatic EDA deletion rule",
        "behavioral item-to-item links cannot be built for items without events",
        "low event volume can make pair scores unreliable",
        "popular items can dominate recommendations without normalization",
        "raw pair count can reflect popularity rather than similarity",
        "evidence from one or a few users can be noisy",
    ],
})

display(item_preprocessing_decisions)


condition,decision,reason
str,str,str
"""item_id is not in product cata…","""exclude from recommendation ca…","""catalog metadata is required t…"
"""item_id is not in product cata…","""needs manual preprocessing dec…","""removing behavioral rows can c…"
"""catalog item has no behavioral…","""use fallback instead of co-vis…","""behavioral item-to-item links …"
"""item has very few behavioral e…","""keep as candidate, but rely on…","""low event volume can make pair…"
"""item is extremely popular""","""keep, but normalize popularity…","""popular items can dominate rec…"
"""pair contains very popular ite…","""apply popularity normalization…","""raw pair count can reflect pop…"
"""item has too few unique users""","""keep only with fallback or req…","""evidence from one or a few use…"


### Item analysis conclusions

- В товарных событиях найдено больше уникальных `item_id`, чем есть в справочнике `product_information`.
- Значительная часть `item_id`, встречающихся в поведении, отсутствует в справочнике; при этом их доля в общем числе товарных событий мала.
- Для MVP товары вне справочника нужно исключить из recommendation candidates, потому что для них нет названия, бренда, типа и категории.
- Распределение событий по товарам сильно скошено: есть выраженный popularity bias и long-tail.
- Raw pair count будет завышать score для популярных товаров, поэтому для baseline нужна popularity normalization.
- Для товаров без поведения и для long-tail нужен fallback, например по `category_id`, а позже — content-based fallback по названию, типу и бренду.
- Точный глобальный `unique_users_per_item` намеренно не считается в EDA из-за размера `user_actions`; sample-day оценка используется как безопасная диагностическая проверка.


## 11. Search query analysis

Раздел анализирует `search_query`.

Этот блок не нужен для первого item-to-item co-visitation baseline напрямую, потому что `search` не содержит `item_id`. Но он важен для будущего co-search сигнала.

Цель:
- оценить, сколько поисковых событий пригодны для анализа;
- посчитать уникальные нормализованные поисковые запросы;
- найти частые, редкие и потенциально мусорные запросы;
- проверить, можно ли связать поисковый запрос с последующим `click` или `to_cart`.


In [52]:
def normalized_search_query_expr() -> pl.Expr:
    """Normalize search_query for EDA-level aggregation."""
    return (
        pl.col("search_query")
        .str.to_lowercase()
        .str.strip_chars()
        .str.replace_all(r"\s+", " ")
        .alias("normalized_query")
    )


ANOMALOUS_USER_IDS = [6195822]
SESSION_TIMEOUT_SECONDS = 30 * 60


In [53]:
search_missing_row = (
    missing_by_action_type
    .filter(pl.col("action_type") == "search")
    .select([
        "rows",
        "search_query_null_count",
        "search_query_null_share",
        "search_query_blank_count",
        "search_query_blank_share",
    ])
    .with_columns([
        (
            pl.col("rows")
            - pl.col("search_query_null_count")
            - pl.col("search_query_blank_count")
        ).alias("usable_search_events"),
        (
            (
                pl.col("rows")
                - pl.col("search_query_null_count")
                - pl.col("search_query_blank_count")
            )
            / pl.col("rows")
        ).alias("usable_search_events_share"),
    ])
)

display(search_missing_row)


rows,search_query_null_count,search_query_null_share,search_query_blank_count,search_query_blank_share,usable_search_events,usable_search_events_share
i64,i64,f64,i64,f64,i64,f64
32274194,84215,0.002609,773,0.000024,32189206,0.997367


In [54]:
SEARCH_QUERY_COUNTS_PATH = CACHE_DIR / "eda_search_query_counts.parquet"

if should_load_eda_cache(SEARCH_QUERY_COUNTS_PATH, "search_query_counts"):
    search_query_counts = read_eda_cache(SEARCH_QUERY_COUNTS_PATH, "search_query_counts")
else:
    search_query_frames = []

    search_files = (
        file_inventory
        .filter(pl.col("action_type") == "search")
        .sort("date")
    )

    for i, row in enumerate(search_files.iter_rows(named=True), start=1):
        search_query_counts_partition = (
            pl.scan_parquet(
                row["path"],
                hive_partitioning=True,
            )
            .select([
                "user_id",
                "search_query",
                normalized_search_query_expr(),
            ])
            .filter(
                pl.col("normalized_query").is_not_null()
                & (pl.col("normalized_query") != "")
            )
            .group_by("normalized_query")
            .agg([
                pl.len().alias("events"),
                pl.col("user_id").n_unique().alias("unique_users_in_partition"),
            ])
            .with_columns([
                pl.lit(row["date"]).alias("date"),
            ])
            .collect(engine="streaming")
        )

        search_query_frames.append(search_query_counts_partition)

        if i % 10 == 0:
            print(f"Processed {i} / {search_files.height} search partitions")

    search_query_counts_by_day = pl.concat(search_query_frames)

    search_query_counts = (
        search_query_counts_by_day
        .group_by("normalized_query")
        .agg([
            pl.col("events").sum().alias("events"),
            pl.col("date").n_unique().alias("days_with_query"),
            pl.col("unique_users_in_partition").sum().alias("user_day_count_approx"),
        ])
        .with_columns([
            pl.col("normalized_query").str.len_chars().alias("query_chars"),
            pl.col("normalized_query").str.split(" ").list.len().alias("query_words"),
        ])
        .sort("events", descending=True)
    )

    write_eda_cache(search_query_counts, SEARCH_QUERY_COUNTS_PATH, "search_query_counts")

display(search_query_counts.head(20))


Processed 10 / 61 search partitions
Processed 20 / 61 search partitions
Processed 30 / 61 search partitions
Processed 40 / 61 search partitions
Processed 50 / 61 search partitions
Processed 60 / 61 search partitions
Saved search_query_counts cache: C:\Users\User\Projects\OZON-Similar-products\outputs\interim\eda_search_query_counts.parquet


normalized_query,events,days_with_query,user_day_count_approx,query_chars,query_words
str,u32,u32,u32,u32,u32
"""молоко""",345750,61,317733,6,1
"""хлеб""",260170,61,239976,4,1
"""сыр""",221862,61,198978,3,1
"""мороженое""",154618,61,135269,9,1
"""яйца""",143808,61,135981,4,1
…,…,…,…,…,…
"""торт""",94606,61,78592,4,1
"""яйца куриные""",92159,61,88483,12,2
"""сосиски""",90091,61,83964,7,1


In [55]:
search_query_counts_summary = search_query_counts.select([
    pl.len().alias("unique_normalized_queries"),
    pl.col("events").sum().alias("search_events_with_query"),

    pl.col("events").mean().alias("mean_events_per_query"),
    pl.col("events").median().alias("median_events_per_query"),
    pl.col("events").quantile(0.90).alias("p90_events_per_query"),
    pl.col("events").quantile(0.95).alias("p95_events_per_query"),
    pl.col("events").quantile(0.99).alias("p99_events_per_query"),
    pl.col("events").max().alias("max_events_per_query"),

    (pl.col("events") == 1).sum().alias("queries_with_1_event"),
    (pl.col("events") <= 5).sum().alias("queries_with_5_or_less_events"),
    (pl.col("events") <= 10).sum().alias("queries_with_10_or_less_events"),
])

search_query_counts_summary = search_query_counts_summary.with_columns([
    (pl.col("queries_with_1_event") / pl.col("unique_normalized_queries")).alias("queries_with_1_event_share"),
    (pl.col("queries_with_5_or_less_events") / pl.col("unique_normalized_queries")).alias("queries_with_5_or_less_events_share"),
    (pl.col("queries_with_10_or_less_events") / pl.col("unique_normalized_queries")).alias("queries_with_10_or_less_events_share"),
])

display(search_query_counts_summary)


unique_normalized_queries,search_events_with_query,mean_events_per_query,median_events_per_query,p90_events_per_query,p95_events_per_query,p99_events_per_query,max_events_per_query,queries_with_1_event,queries_with_5_or_less_events,queries_with_10_or_less_events,queries_with_1_event_share,queries_with_5_or_less_events_share,queries_with_10_or_less_events_share
u32,u32,f64,f64,f64,f64,f64,u32,u32,u32,u32,f64,f64,f64
2761580,32189206,11.656083,1.0,5.0,11.0,81.0,345750,1839051,2509496,2616446,0.665942,0.908717,0.947445


In [56]:
top_search_queries = (
    search_query_counts
    .select([
        "normalized_query",
        "events",
        "days_with_query",
        "user_day_count_approx",
        "query_chars",
        "query_words",
    ])
    .head(50)
)

display(top_search_queries)


normalized_query,events,days_with_query,user_day_count_approx,query_chars,query_words
str,u32,u32,u32,u32,u32
"""молоко""",345750,61,317733,6,1
"""хлеб""",260170,61,239976,4,1
"""сыр""",221862,61,198978,3,1
"""мороженое""",154618,61,135269,9,1
"""яйца""",143808,61,135981,4,1
…,…,…,…,…,…
"""цветы""",45423,61,39550,5,1
"""помидоры свежие""",45061,61,43385,15,2
"""энергетик""",44352,61,38082,9,1


In [57]:
search_query_length_summary = search_query_counts.select([
    pl.len().alias("unique_normalized_queries"),

    pl.col("query_chars").mean().alias("mean_query_chars"),
    pl.col("query_chars").median().alias("median_query_chars"),
    pl.col("query_chars").quantile(0.90).alias("p90_query_chars"),
    pl.col("query_chars").quantile(0.99).alias("p99_query_chars"),
    pl.col("query_chars").max().alias("max_query_chars"),

    pl.col("query_words").mean().alias("mean_query_words"),
    pl.col("query_words").median().alias("median_query_words"),
    pl.col("query_words").quantile(0.90).alias("p90_query_words"),
    pl.col("query_words").quantile(0.99).alias("p99_query_words"),
    pl.col("query_words").max().alias("max_query_words"),
])

display(search_query_length_summary)


unique_normalized_queries,mean_query_chars,median_query_chars,p90_query_chars,p99_query_chars,max_query_chars,mean_query_words,median_query_words,p90_query_words,p99_query_words,max_query_words
u32,f64,f64,f64,f64,u32,f64,f64,f64,f64,u32
2761580,21.033102,19.0,34.0,54.0,1000,3.207052,3.0,5.0,8.0,156


In [58]:
search_query_quality_flags = search_query_counts.select([
    pl.len().alias("unique_normalized_queries"),
    pl.col("events").sum().alias("events"),

    (pl.col("query_chars") <= 1).sum().alias("queries_with_1_char"),
    pl.when(pl.col("query_chars") <= 1).then(pl.col("events")).otherwise(0).sum().alias("events_with_1_char_query"),

    (pl.col("query_chars") > 100).sum().alias("queries_longer_than_100_chars"),
    pl.when(pl.col("query_chars") > 100).then(pl.col("events")).otherwise(0).sum().alias("events_with_long_query"),

    (pl.col("query_words") > 10).sum().alias("queries_with_more_than_10_words"),
    pl.when(pl.col("query_words") > 10).then(pl.col("events")).otherwise(0).sum().alias("events_with_more_than_10_words"),

    pl.col("normalized_query").str.contains(r"^\d+$").sum().alias("numeric_only_queries"),
    pl.when(pl.col("normalized_query").str.contains(r"^\d+$")).then(pl.col("events")).otherwise(0).sum().alias("events_with_numeric_only_query"),
])

search_query_quality_flags = search_query_quality_flags.with_columns([
    (pl.col("queries_with_1_char") / pl.col("unique_normalized_queries")).alias("queries_with_1_char_share"),
    (pl.col("events_with_1_char_query") / pl.col("events")).alias("events_with_1_char_query_share"),

    (pl.col("queries_longer_than_100_chars") / pl.col("unique_normalized_queries")).alias("queries_longer_than_100_chars_share"),
    (pl.col("events_with_long_query") / pl.col("events")).alias("events_with_long_query_share"),

    (pl.col("queries_with_more_than_10_words") / pl.col("unique_normalized_queries")).alias("queries_with_more_than_10_words_share"),
    (pl.col("events_with_more_than_10_words") / pl.col("events")).alias("events_with_more_than_10_words_share"),

    (pl.col("numeric_only_queries") / pl.col("unique_normalized_queries")).alias("numeric_only_queries_share"),
    (pl.col("events_with_numeric_only_query") / pl.col("events")).alias("events_with_numeric_only_query_share"),
])

display(search_query_quality_flags)


unique_normalized_queries,events,queries_with_1_char,events_with_1_char_query,queries_longer_than_100_chars,events_with_long_query,queries_with_more_than_10_words,events_with_more_than_10_words,numeric_only_queries,events_with_numeric_only_query,queries_with_1_char_share,events_with_1_char_query_share,queries_longer_than_100_chars_share,events_with_long_query_share,queries_with_more_than_10_words_share,events_with_more_than_10_words_share,numeric_only_queries_share,events_with_numeric_only_query_share
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,f64,f64,f64,f64,f64,f64,f64,f64
2761580,32189206,223,10895,3462,4123,11066,14026,17296,36778,0.000081,0.000338,0.001254,0.000128,0.004007,0.000436,0.006263,0.001143


In [59]:
from datetime import timedelta

sample_search_events = (
    sample_day_lf
    .filter(
        (pl.col("action_type") == "search")
        & (~pl.col("user_id").is_in(ANOMALOUS_USER_IDS))
    )
    .select([
        "user_id",
        "timestamp",
        "search_query",
        normalized_search_query_expr(),
    ])
    .filter(
        pl.col("normalized_query").is_not_null()
        & (pl.col("normalized_query") != "")
    )
    .sort(["user_id", "timestamp"])
    .collect()
)

sample_item_actions_for_search = (
    sample_day_lf
    .filter(
        pl.col("action_type").is_in(["click", "to_cart"])
        & pl.col("item_id").is_not_null()
        & (~pl.col("user_id").is_in(ANOMALOUS_USER_IDS))
    )
    .select([
        "user_id",
        "timestamp",
        "action_type",
        "item_id",
    ])
    .sort(["user_id", "timestamp"])
    .collect()
)

sample_query_item_links = (
    sample_item_actions_for_search
    .join_asof(
        sample_search_events.select(["user_id", "timestamp", "normalized_query"]),
        on="timestamp",
        by="user_id",
        strategy="backward",
        tolerance=timedelta(minutes=30),
    )
    .filter(pl.col("normalized_query").is_not_null())
)

total_click_to_cart_events = sample_item_actions_for_search.height

sample_query_item_link_summary = pl.DataFrame({
    "metric": [
        "sample_search_events",
        "sample_click_to_cart_events",
        "click_to_cart_events_linked_to_previous_search_30m",
    ],
    "value": [
        sample_search_events.height,
        total_click_to_cart_events,
        sample_query_item_links.height,
    ],
}).with_columns(
    pl.when(
        pl.col("metric") == "click_to_cart_events_linked_to_previous_search_30m"
    )
    .then(pl.col("value") / pl.lit(total_click_to_cart_events))
    .otherwise(None)
    .alias("share_of_click_to_cart_events")
)

display(sample_query_item_link_summary)
display(sample_query_item_links.head(20))


C:\Users\User\AppData\Local\Temp\ipykernel_5416\4135329724.py:42: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  .join_asof(


metric,value,share_of_click_to_cart_events
str,i64,f64
"""sample_search_events""",424315,null
"""sample_click_to_cart_events""",408619,null
"""click_to_cart_events_linked_to…",387894,0.94928


user_id,timestamp,action_type,item_id,normalized_query
i32,datetime[ns],str,i32,str
100,2024-04-24 16:38:21,"""click""",138984,"""масло подсолнечное нерафиниров…"
170,2024-04-24 15:56:04,"""click""",105223,"""хлеб"""
170,2024-04-24 15:56:51,"""to_cart""",117230,"""хлеб"""
170,2024-04-24 15:57:42,"""click""",60218,"""рис"""
170,2024-04-24 16:00:40,"""click""",138679,"""тарталетки"""
…,…,…,…,…
350,2024-04-24 13:33:55,"""click""",65777,"""томаты свежие"""
350,2024-04-24 13:35:09,"""to_cart""",165881,"""черноголовка лимонад"""
350,2024-04-24 13:36:01,"""to_cart""",168516,"""черноголовка 2л"""


In [60]:
top_query_item_links_sample = (
    sample_query_item_links
    .group_by(["normalized_query", "item_id", "action_type"])
    .agg(pl.len().alias("events"))
    .join(products, on="item_id", how="left")
    .select([
        "normalized_query",
        "item_id",
        "name",
        "brand",
        "type",
        "category_id",
        "category_name",
        "action_type",
        "events",
    ])
    .sort("events", descending=True)
    .head(50)
)

display(top_query_item_links_sample)


normalized_query,item_id,name,brand,type,category_id,category_name,action_type,events
str,i32,str,str,str,i32,str,str,u32
"""молоко""",214829,"""Молоко пастеризованное 2,5% 93…","""Простоквашино""","""Молоко""",248,"""Пастеризованное молоко""","""to_cart""",656
"""молоко""",114040,"""Молоко питьевое ультрапастериз…","""Село Зеленое""","""Молоко""",814,"""Стерилизованное молоко""","""to_cart""",604
"""хлеб""",60173,"""Батон ""Нарезной"" 400 г, Коломе…","""Коломенский""","""Хлеб""",848,"""Белый хлеб""","""to_cart""",594
"""туалетная бумага""",11306,"""Туалетная бумага Zewa Natural …","""Zewa""","""Туалетная бумага""",117,"""Туалетная бумага""","""to_cart""",392
"""огурцы""",243771,"""Огурцы Кураж, 400 г, Россия""","""Нет бренда""","""Овощи""",782,"""Огурцы""","""to_cart""",383
…,…,…,…,…,…,…,…,…
"""томатная паста""",242738,"""Томатная паста Помидорка, 70 г""","""Помидорка""","""Томатная паста""",813,"""Томатная паста, кетчуп""","""to_cart""",139
"""огурцы""",192864,"""Огурцы Геракл 500 г, хрустящие…","""Нет бренда""","""Овощи""",782,"""Огурцы""","""to_cart""",138
"""яйца""",96996,"""Яйца куриные отборные СО 10 шт…","""Ozon fresh""","""Яйца""",838,"""Яйца""","""to_cart""",138


In [61]:
search_preprocessing_decisions = pl.DataFrame({
    "condition": [
        "action_type = search and search_query is null",
        "action_type = search and normalized search_query is empty",
        "normalized search_query is very rare",
        "normalized search_query is too short",
        "normalized search_query is too long",
        "click/to_cart has previous search within 30 minutes",
    ],
    "decision": [
        "drop from co-search analysis",
        "drop from co-search analysis",
        "keep for EDA, but use min_query_events for co-search",
        "consider filtering or downweighting",
        "consider filtering or manual inspection",
        "can be used as query -> item signal candidate",
    ],
    "reason": [
        "нет текстового интента",
        "нет содержательного текстового интента",
        "редкие запросы дают шумные связи",
        "слишком короткий запрос часто неоднозначен",
        "слишком длинный запрос может быть мусорным или вставленным текстом",
        "есть временная связь между поиском и товарным действием",
    ],
})

display(search_preprocessing_decisions)


condition,decision,reason
str,str,str
"""action_type = search and searc…","""drop from co-search analysis""","""нет текстового интента"""
"""action_type = search and norma…","""drop from co-search analysis""","""нет содержательного текстового…"
"""normalized search_query is ver…","""keep for EDA, but use min_quer…","""редкие запросы дают шумные свя…"
"""normalized search_query is too…","""consider filtering or downweig…","""слишком короткий запрос часто …"
"""normalized search_query is too…","""consider filtering or manual i…","""слишком длинный запрос может б…"
"""click/to_cart has previous sea…","""can be used as query -> item s…","""есть временная связь между пои…"


### Search query analysis conclusions

- В данных найдено 32 274 194 search-событий.
- Для анализа пригодны 32 189 206 search-событий, то есть около 99.74%: `search_query` почти всегда заполнен.
- После простой нормализации найдено 2 761 580 уникальных поисковых запросов.
- Распределение запросов сильно long-tail: медиана — 1 событие на запрос, 66.59% запросов встречаются только один раз, 90.87% — не больше 5 раз.
- Топ запросов выглядят как реальные пользовательские интенты: например, `молоко`, `хлеб`, `сыр`, `мороженое`, `яйца`.
- Потенциально мусорные запросы есть, но их доля в событиях мала: односимвольные запросы дают около 0.03% search-событий, слишком длинные запросы — около 0.01%, numeric-only — около 0.11%.
- На `sample_day = 2024-04-24` после исключения аномального пользователя `6195822` около 94.93% `click/to_cart` событий имеют предшествующее search-событие того же пользователя в 30-минутном окне. Это диагностическая временная связь, а не доказательство причинности.
- На `sample_day = 2024-04-24` топ связок `query -> item` выглядят смыслово корректно: например, `молоко` ведет к товарам категории молока, `хлеб` — к хлебу, `огурцы` — к огурцам.
- Co-search не включаем в первый item-to-item baseline, но оставляем как перспективное улучшение.
- Для будущего co-search нужны фильтры: удалять пустые запросы, нормализовать текст, вводить `min_query_events`, осторожно обрабатывать слишком короткие и слишком длинные запросы.


## 12. Timestamp and session feasibility

Раздел проверяет, можно ли строить пользовательские сессии.

Цель:
- убедиться, что события можно упорядочить по `user_id` и `timestamp`;
- проверить интервалы между соседними событиями пользователя;
- оценить, насколько разумен session timeout 30 минут;
- найти долю нулевых, коротких и длинных интервалов между действиями;
- зафиксировать решения для будущей sessionization.


In [62]:
sample_day_session_events = (
    sample_day_lf
    .filter(~pl.col("user_id").is_in(ANOMALOUS_USER_IDS))
    .select([
        "user_id",
        "timestamp",
        "action_type",
        "item_id",
    ])
    .sort(["user_id", "timestamp"])
    .collect()
)

display(sample_day_session_events.head(20))
print("Sample-day session events:", sample_day_session_events.height)


user_id,timestamp,action_type,item_id
i32,datetime[ns],str,i32
44,2024-04-24 09:20:33,"""search""",null
44,2024-04-24 09:20:34,"""view""",86148
44,2024-04-24 09:20:34,"""view""",370
100,2024-04-24 16:37:57,"""search""",null
100,2024-04-24 16:37:57,"""view""",80030
…,…,…,…
100,2024-04-24 16:38:17,"""view""",202534
100,2024-04-24 16:38:18,"""view""",75351
100,2024-04-24 16:38:18,"""view""",22315


Sample-day session events: 7171520


In [63]:
sample_day_event_gaps = (
    sample_day_session_events
    .with_columns([
        pl.col("timestamp").shift(1).over("user_id").alias("previous_timestamp"),
    ])
    .with_columns([
        (
            pl.col("timestamp") - pl.col("previous_timestamp")
        ).dt.total_seconds().alias("gap_seconds")
    ])
    .filter(pl.col("gap_seconds").is_not_null())
)

display(sample_day_event_gaps.head(20))


user_id,timestamp,action_type,item_id,previous_timestamp,gap_seconds
i32,datetime[ns],str,i32,datetime[ns],i64
44,2024-04-24 09:20:34,"""view""",86148,2024-04-24 09:20:33,1
44,2024-04-24 09:20:34,"""view""",370,2024-04-24 09:20:34,0
100,2024-04-24 16:37:57,"""view""",80030,2024-04-24 16:37:57,0
100,2024-04-24 16:38:14,"""view""",104735,2024-04-24 16:37:57,17
100,2024-04-24 16:38:14,"""view""",211252,2024-04-24 16:38:14,0
…,…,…,…,…,…
100,2024-04-24 16:38:18,"""view""",22315,2024-04-24 16:38:18,0
100,2024-04-24 16:38:18,"""view""",181550,2024-04-24 16:38:18,0
100,2024-04-24 16:38:18,"""view""",138984,2024-04-24 16:38:18,0


In [64]:
sample_day_gap_summary = sample_day_event_gaps.select([
    pl.len().alias("event_gaps"),

    pl.col("gap_seconds").min().alias("min_gap_seconds"),
    pl.col("gap_seconds").mean().alias("mean_gap_seconds"),
    pl.col("gap_seconds").median().alias("median_gap_seconds"),
    pl.col("gap_seconds").quantile(0.75).alias("p75_gap_seconds"),
    pl.col("gap_seconds").quantile(0.90).alias("p90_gap_seconds"),
    pl.col("gap_seconds").quantile(0.95).alias("p95_gap_seconds"),
    pl.col("gap_seconds").quantile(0.99).alias("p99_gap_seconds"),
    pl.col("gap_seconds").max().alias("max_gap_seconds"),

    (pl.col("gap_seconds") < 0).sum().alias("negative_gaps"),
    (pl.col("gap_seconds") == 0).sum().alias("zero_gaps"),
    (pl.col("gap_seconds") <= 60).sum().alias("gaps_1_min_or_less"),
    (pl.col("gap_seconds") <= 300).sum().alias("gaps_5_min_or_less"),
    (pl.col("gap_seconds") <= 1800).sum().alias("gaps_30_min_or_less"),
    (pl.col("gap_seconds") > 1800).sum().alias("gaps_more_than_30_min"),
])

sample_day_gap_summary = sample_day_gap_summary.with_columns([
    (pl.col("negative_gaps") / pl.col("event_gaps")).alias("negative_gaps_share"),
    (pl.col("zero_gaps") / pl.col("event_gaps")).alias("zero_gaps_share"),
    (pl.col("gaps_30_min_or_less") / pl.col("event_gaps")).alias("gaps_30_min_or_less_share"),
    (pl.col("gaps_more_than_30_min") / pl.col("event_gaps")).alias("gaps_more_than_30_min_share"),
])

display(sample_day_gap_summary)


event_gaps,min_gap_seconds,mean_gap_seconds,median_gap_seconds,p75_gap_seconds,p90_gap_seconds,p95_gap_seconds,p99_gap_seconds,max_gap_seconds,negative_gaps,zero_gaps,gaps_1_min_or_less,gaps_5_min_or_less,gaps_30_min_or_less,gaps_more_than_30_min,negative_gaps_share,zero_gaps_share,gaps_30_min_or_less_share,gaps_more_than_30_min_share
u32,i64,f64,f64,f64,f64,f64,f64,i64,u32,u32,u32,u32,u32,u32,f64,f64,f64,f64
7048742,0,49.199306,0.0,1.0,6.0,13.0,133.0,84092,0,4229372,6934922,7002842,7025442,23300,0.0,0.600018,0.996694,0.003306


In [65]:
sample_day_multi_event_user_sessions = (
    sample_day_event_gaps
    .with_columns([
        (
            (pl.col("gap_seconds") > SESSION_TIMEOUT_SECONDS)
            .cast(pl.Int64)
        ).alias("is_new_session_after_gap")
    ])
)

sample_day_multi_event_user_session_counts = (
    sample_day_multi_event_user_sessions
    .group_by("user_id")
    .agg([
        (pl.col("is_new_session_after_gap").sum() + 1).alias("sessions"),
        pl.len().alias("events_after_first"),
    ])
    .with_columns([
        (pl.col("events_after_first") + 1).alias("events"),
    ])
)

sample_day_multi_event_user_session_count_summary = sample_day_multi_event_user_session_counts.select([
    pl.len().alias("users_with_2_plus_events"),
    pl.col("sessions").sum().alias("estimated_sessions_for_users_with_2_plus_events"),
    pl.col("sessions").mean().alias("mean_sessions_per_multi_event_user"),
    pl.col("sessions").median().alias("median_sessions_per_multi_event_user"),
    pl.col("sessions").quantile(0.90).alias("p90_sessions_per_multi_event_user"),
    pl.col("sessions").quantile(0.99).alias("p99_sessions_per_multi_event_user"),
    pl.col("sessions").max().alias("max_sessions_per_multi_event_user"),
])

display(sample_day_multi_event_user_session_count_summary)


users_with_2_plus_events,estimated_sessions_for_users_with_2_plus_events,mean_sessions_per_multi_event_user,median_sessions_per_multi_event_user,p90_sessions_per_multi_event_user,p99_sessions_per_multi_event_user,max_sessions_per_multi_event_user
u32,i64,f64,f64,f64,f64,i64
113328,136628,1.205598,1.0,2.0,3.0,12


In [66]:
session_preprocessing_decisions = pl.DataFrame({
    "condition": [
        "timestamp is null",
        "events are sortable by user_id and timestamp",
        "gap between user events > 30 minutes",
        "gap between user events <= 30 minutes",
        "session has too many events",
        "session has too many unique items",
    ],
    "decision": [
        "drop",
        "use timestamp ordering for sessionization",
        "start new session",
        "keep within the same session",
        "truncate or split session during preprocessing",
        "truncate or cap pair generation during preprocessing",
    ],
    "reason": [
        "невозможно определить порядок действий",
        "можно построить последовательность действий пользователя",
        "длинный разрыв обычно означает новый пользовательский контекст",
        "короткий разрыв вероятно относится к одному пользовательскому контексту",
        "слишком длинная сессия создает много шумных item-pairs",
        "слишком много товаров в сессии создает комбинаторный взрыв пар",
    ],
})

display(session_preprocessing_decisions)


condition,decision,reason
str,str,str
"""timestamp is null""","""drop""","""невозможно определить порядок …"
"""events are sortable by user_id…","""use timestamp ordering for ses…","""можно построить последовательн…"
"""gap between user events > 30 m…","""start new session""","""длинный разрыв обычно означает…"
"""gap between user events <= 30 …","""keep within the same session""","""короткий разрыв вероятно относ…"
"""session has too many events""","""truncate or split session duri…","""слишком длинная сессия создает…"
"""session has too many unique it…","""truncate or cap pair generatio…","""слишком много товаров в сессии…"


In [67]:
sample_day_events_with_sessions = (
    sample_day_session_events
    .with_columns([
        pl.col("timestamp").shift(1).over("user_id").alias("previous_timestamp"),
    ])
    .with_columns([
        (
            pl.col("timestamp") - pl.col("previous_timestamp")
        ).dt.total_seconds().alias("gap_seconds")
    ])
    .with_columns([
        (
            pl.col("previous_timestamp").is_null()
            | (pl.col("gap_seconds") > SESSION_TIMEOUT_SECONDS)
        ).cast(pl.Int64).alias("is_new_session")
    ])
    .with_columns([
        pl.col("is_new_session").cum_sum().over("user_id").alias("session_number")
    ])
)

display(sample_day_events_with_sessions.head(20))


user_id,timestamp,action_type,item_id,previous_timestamp,gap_seconds,is_new_session,session_number
i32,datetime[ns],str,i32,datetime[ns],i64,i64,i64
44,2024-04-24 09:20:33,"""search""",null,null,null,1,1
44,2024-04-24 09:20:34,"""view""",86148,2024-04-24 09:20:33,1,0,1
44,2024-04-24 09:20:34,"""view""",370,2024-04-24 09:20:34,0,0,1
100,2024-04-24 16:37:57,"""search""",null,null,null,1,1
100,2024-04-24 16:37:57,"""view""",80030,2024-04-24 16:37:57,0,0,1
…,…,…,…,…,…,…,…
100,2024-04-24 16:38:17,"""view""",202534,2024-04-24 16:38:17,0,0,1
100,2024-04-24 16:38:18,"""view""",75351,2024-04-24 16:38:17,1,0,1
100,2024-04-24 16:38:18,"""view""",22315,2024-04-24 16:38:18,0,0,1


In [68]:
sample_day_session_level = (
    sample_day_events_with_sessions
    .group_by(["user_id", "session_number"])
    .agg([
        pl.len().alias("events"),
        pl.col("item_id").drop_nulls().n_unique().alias("unique_items"),
        pl.col("timestamp").min().alias("session_start"),
        pl.col("timestamp").max().alias("session_end"),
    ])
    .with_columns([
        (
            pl.col("session_end") - pl.col("session_start")
        ).dt.total_seconds().alias("session_duration_seconds")
    ])
)

sample_day_session_level_summary = sample_day_session_level.select([
    pl.len().alias("sessions"),
    pl.col("user_id").n_unique().alias("users"),
    pl.col("events").sum().alias("events"),

    pl.col("events").mean().alias("mean_events_per_session"),
    pl.col("events").median().alias("median_events_per_session"),
    pl.col("events").quantile(0.90).alias("p90_events_per_session"),
    pl.col("events").quantile(0.95).alias("p95_events_per_session"),
    pl.col("events").quantile(0.99).alias("p99_events_per_session"),
    pl.col("events").max().alias("max_events_per_session"),

    pl.col("unique_items").mean().alias("mean_unique_items_per_session"),
    pl.col("unique_items").median().alias("median_unique_items_per_session"),
    pl.col("unique_items").quantile(0.90).alias("p90_unique_items_per_session"),
    pl.col("unique_items").quantile(0.95).alias("p95_unique_items_per_session"),
    pl.col("unique_items").quantile(0.99).alias("p99_unique_items_per_session"),
    pl.col("unique_items").max().alias("max_unique_items_per_session"),

    pl.col("session_duration_seconds").median().alias("median_session_duration_seconds"),
    pl.col("session_duration_seconds").quantile(0.90).alias("p90_session_duration_seconds"),
    pl.col("session_duration_seconds").quantile(0.99).alias("p99_session_duration_seconds"),
    pl.col("session_duration_seconds").max().alias("max_session_duration_seconds"),
])

display(sample_day_session_level_summary)
display(sample_day_session_level.sort("events", descending=True).head(20))


sessions,users,events,mean_events_per_session,median_events_per_session,p90_events_per_session,p95_events_per_session,p99_events_per_session,max_events_per_session,mean_unique_items_per_session,median_unique_items_per_session,p90_unique_items_per_session,p95_unique_items_per_session,p99_unique_items_per_session,max_unique_items_per_session,median_session_duration_seconds,p90_session_duration_seconds,p99_session_duration_seconds,max_session_duration_seconds
u32,u32,u32,f64,f64,f64,f64,f64,u32,f64,f64,f64,f64,f64,u32,f64,f64,f64,i64
146078,122778,7171520,49.093772,21.0,128.0,191.0,363.0,2919,37.831672,16.0,100.0,150.0,284.0,1822,53.0,800.0,2427.0,10274


user_id,session_number,events,unique_items,session_start,session_end,session_duration_seconds
i32,i64,u32,u32,datetime[ns],datetime[ns],i64
4809133,2,2919,1822,2024-04-24 06:24:21,2024-04-24 07:59:58,5737
9421240,1,2902,46,2024-04-24 07:14:27,2024-04-24 10:05:41,10274
8496361,1,1801,1390,2024-04-24 00:51:55,2024-04-24 02:53:15,7280
10207636,1,1796,1521,2024-04-24 09:45:35,2024-04-24 10:35:53,3018
8682416,1,1628,1159,2024-04-24 17:04:57,2024-04-24 17:34:27,1770
…,…,…,…,…,…,…
2480511,1,1136,696,2024-04-24 17:51:26,2024-04-24 18:09:33,1087
436110,1,1107,945,2024-04-24 07:57:55,2024-04-24 08:31:25,2010
7781389,2,1089,668,2024-04-24 20:33:49,2024-04-24 21:25:26,3097


In [69]:
sample_day_session_thresholds = sample_day_session_level.select([
    pl.len().alias("sessions"),

    (pl.col("events") >= 50).sum().alias("sessions_with_50_plus_events"),
    (pl.col("events") >= 100).sum().alias("sessions_with_100_plus_events"),
    (pl.col("events") >= 200).sum().alias("sessions_with_200_plus_events"),
    (pl.col("events") >= 500).sum().alias("sessions_with_500_plus_events"),

    (pl.col("unique_items") >= 50).sum().alias("sessions_with_50_plus_unique_items"),
    (pl.col("unique_items") >= 100).sum().alias("sessions_with_100_plus_unique_items"),
    (pl.col("unique_items") >= 200).sum().alias("sessions_with_200_plus_unique_items"),
])

sample_day_session_thresholds = sample_day_session_thresholds.with_columns([
    (pl.col("sessions_with_50_plus_events") / pl.col("sessions")).alias("sessions_with_50_plus_events_share"),
    (pl.col("sessions_with_100_plus_events") / pl.col("sessions")).alias("sessions_with_100_plus_events_share"),
    (pl.col("sessions_with_200_plus_events") / pl.col("sessions")).alias("sessions_with_200_plus_events_share"),
    (pl.col("sessions_with_500_plus_events") / pl.col("sessions")).alias("sessions_with_500_plus_events_share"),

    (pl.col("sessions_with_50_plus_unique_items") / pl.col("sessions")).alias("sessions_with_50_plus_unique_items_share"),
    (pl.col("sessions_with_100_plus_unique_items") / pl.col("sessions")).alias("sessions_with_100_plus_unique_items_share"),
    (pl.col("sessions_with_200_plus_unique_items") / pl.col("sessions")).alias("sessions_with_200_plus_unique_items_share"),
])

display(sample_day_session_thresholds)


sessions,sessions_with_50_plus_events,sessions_with_100_plus_events,sessions_with_200_plus_events,sessions_with_500_plus_events,sessions_with_50_plus_unique_items,sessions_with_100_plus_unique_items,sessions_with_200_plus_unique_items,sessions_with_50_plus_events_share,sessions_with_100_plus_events_share,sessions_with_200_plus_events_share,sessions_with_500_plus_events_share,sessions_with_50_plus_unique_items_share,sessions_with_100_plus_unique_items_share,sessions_with_200_plus_unique_items_share
u32,u32,u32,u32,u32,u32,u32,u32,f64,f64,f64,f64,f64,f64,f64
146078,42430,20638,6684,508,33786,14696,3963,0.290461,0.141281,0.045756,0.003478,0.231287,0.100604,0.027129


### Timestamp and session feasibility conclusions

- Глобально проверено, что `timestamp` распарсен и `date` совпадает с датой из `timestamp` по всем partition-level aggregates.
- На `sample_day = 2024-04-24` после сортировки по `user_id` и `timestamp` события можно использовать для временного упорядочивания; это не является проверкой исходного порядка строк.
- На `sample_day = 2024-04-24` после исключения аномального пользователя `6195822` около 99.67% соседних событий находятся в пределах 30 минут.
- Timeout 30 минут остается стартовой MVP-гипотезой по `sample_day = 2024-04-24`, а не глобально валидированным порогом: если между событиями пользователя прошло больше 30 минут, начинаем новую сессию. Перед финализацией порога нужна проверка на нескольких днях.
- Около 60% соседних событий на `sample_day = 2024-04-24` имеют одинаковый timestamp с точностью до секунды. Порядок внутри одинакового timestamp не гарантирован; для co-visitation baseline это ограничение допустимо, потому что baseline не требует строгого порядка действий внутри секунды.
- На `sample_day = 2024-04-24` получилось 146 078 сессий для 122 778 пользователей.
- На `sample_day = 2024-04-24` типичная сессия умеренная: медиана — 21 событие и 16 уникальных товаров.
- На `sample_day = 2024-04-24` есть длинный хвост: p99 — 363 события и 284 уникальных товара на сессию; максимум — 2 919 событий и 1 822 уникальных товара.
- По `sample_day = 2024-04-24` виден риск длинных сессий; для preprocessing стоит предусмотреть cap/split или ограничение генерации пар внутри них.
- Для MVP можно использовать `session_timeout_minutes = 30` и ограничения на максимальное число событий или уникальных товаров в сессии только как стартовые EDA-гипотезы; конкретные пороги нужно проверить за пределами sample-day.


## 13. Baseline readiness checklist

Раздел фиксирует готовность датасета к preprocessing и первому co-visitation baseline.

Чеклист нужен не для расчета новых агрегатов, а для итогового контроля: все ли ключевые риски EDA уже классифицированы и есть ли решения для следующего шага.


In [70]:
baseline_readiness_checklist = pl.DataFrame({
    "check": [
        "Data is loaded",
        "Expected schemas are present",
        "Product item_id is unique",
        "Timestamp is parsed",
        "Date partitions are valid",
        "date matches timestamp date",
        "Action types are known",
        "Co-visitation action types are selected",
        "Search handling is defined",
        "Missing values are classified",
        "Critical missing values in item events are absent",
        "Anomalous users are identified",
        "Popularity bias is identified",
        "Out-of-catalog items are identified",
        "Long-tail items are identified",
        "Session timeout candidate is selected",
        "Long-session risk is identified on sample-day",
        "Baseline can proceed",
    ],
    "status": [
        "OK",
        "OK",
        "OK",
        "OK",
        "OK",
        "OK",
        "OK",
        "OK",
        "OK",
        "OK",
        "OK",
        "OK",
        "OK",
        "OK",
        "OK",
        "OK as sample-day hypothesis",
        "OK as sample-day diagnostic",
        "OK with preprocessing constraints",
    ],
    "evidence": [
        "product_information and user_actions parquet files are found",
        "all expected columns are present in product_information and user_actions",
        "130 035 rows and 130 035 unique item_id",
        "timestamp column is Datetime and has valid daily ranges",
        "61 days, 5 action_type partitions per day",
        "date_timestamp_mismatch_count = 0 for all days",
        "view, search, to_cart, click, favorite",
        "view, click, favorite, to_cart",
        "search is excluded from item-to-item graph and kept for future co-search",
        "missing values analyzed by action_type",
        "item_id is present for view/click/favorite/to_cart",
        "user_id = 6195822 dominates all 61 days as top active user",
        "top items collect a large share of item events",
        "88 425 behavior item_id are not found in product catalog",
        "many items have 1 to 5 behavioral events",
        "30 minutes based on sample_day = 2024-04-24",
        "sample_day = 2024-04-24 sessions have long tail by events and unique_items",
        "data appears usable after global checks and sample-day session diagnostics",
    ],
    "decision": [
        "proceed",
        "proceed",
        "use item_id as product key",
        "use timestamp for sorting events",
        "daily pipeline is feasible",
        "use partition date safely",
        "use known action mapping",
        "use weighted item events",
        "keep search separately",
        "apply preprocessing rules",
        "no missing-value drop needed for item events",
        "exclude or cap anomalous user-day/session contribution",
        "use popularity normalization",
        "exclude from recommendation candidates for MVP",
        "use fallback for weak graph evidence",
        "use as initial MVP value; validate on more days",
        "cap pair generation in long sessions; validate thresholds on more days",
        "move to MVP preprocessing and co-visitation baseline with session caveats",
    ],
})

display(baseline_readiness_checklist)


check,status,evidence,decision
str,str,str,str
"""Data is loaded""","""OK""","""product_information and user_a…","""proceed"""
"""Expected schemas are present""","""OK""","""all expected columns are prese…","""proceed"""
"""Product item_id is unique""","""OK""","""130 035 rows and 130 035 uniqu…","""use item_id as product key"""
"""Timestamp is parsed""","""OK""","""timestamp column is Datetime a…","""use timestamp for sorting even…"
"""Date partitions are valid""","""OK""","""61 days, 5 action_type partiti…","""daily pipeline is feasible"""
…,…,…,…
"""Out-of-catalog items are ident…","""OK""","""88 425 behavior item_id are no…","""exclude from recommendation ca…"
"""Long-tail items are identified""","""OK""","""many items have 1 to 5 behavio…","""use fallback for weak graph ev…"
"""Session timeout candidate is s…","""OK as sample-day hypothesis""","""30 minutes based on sample_day…","""use as initial MVP value; vali…"


In [71]:
baseline_config_draft = {
    "action_weights": {
        "view": 1.0,
        "click": 2.0,
        "favorite": 2.5,
        "to_cart": 4.0,
    },
    "excluded_action_types_from_item_graph": [
        "search",
    ],
    "sessionization": {
        "session_timeout_minutes": 30,
        "exclude_anomalous_user_ids": [6195822],
        "max_events_per_session": 500,
        "max_unique_items_per_session": 200,
    },
    "items": {
        "exclude_items_not_in_product_catalog_from_candidates": True,
        "use_fallback_for_items_without_behavior": True,
        "use_fallback_for_long_tail_items": True,
    },
    "scoring": {
        "use_popularity_normalization": True,
        "candidate_normalization": "pair_score / sqrt(popularity_i * popularity_j)",
        "require_min_pair_count": True,
        "require_min_unique_users": True,
    },
    "search": {
        "use_in_first_item_to_item_baseline": False,
        "keep_for_future_co_search": True,
        "drop_null_or_blank_queries": True,
        "normalize_query_text": True,
        "use_min_query_events_for_co_search": True,
    },
}

baseline_config_draft


{'action_weights': {'view': 1.0,
  'click': 2.0,
  'favorite': 2.5,
  'to_cart': 4.0},
 'excluded_action_types_from_item_graph': ['search'],
 'sessionization': {'session_timeout_minutes': 30,
  'exclude_anomalous_user_ids': [6195822],
  'max_events_per_session': 500,
  'max_unique_items_per_session': 200},
 'items': {'exclude_items_not_in_product_catalog_from_candidates': True,
  'use_fallback_for_items_without_behavior': True,
  'use_fallback_for_long_tail_items': True},
 'scoring': {'use_popularity_normalization': True,
  'candidate_normalization': 'pair_score / sqrt(popularity_i * popularity_j)',
  'require_min_pair_count': True,
  'require_min_unique_users': True},
 'search': {'use_in_first_item_to_item_baseline': False,
  'keep_for_future_co_search': True,
  'drop_null_or_blank_queries': True,
  'normalize_query_text': True,
  'use_min_query_events_for_co_search': True}}

### Baseline readiness checklist conclusions

- EDA поддерживает переход к preprocessing и построению первого co-visitation baseline при зафиксированных ограничениях.
- Схемы соответствуют ожидаемым, `timestamp` корректен, дневные партиции полные, `date` совпадает с датой из `timestamp`.
- Для item-to-item graph используем только товарные события: `view`, `click`, `favorite`, `to_cart`.
- `search` не включаем в первый item-to-item graph, но сохраняем для будущего co-search.
- Критичных missing values в товарных событиях нет.
- Обязательные preprocessing-ограничения: исключить или ограничить вклад аномального пользователя `6195822`, исключить товары вне справочника из candidates, а также ограничить длинные сессии; вывод по длинным сессиям основан на `sample_day = 2024-04-24`.
- Для baseline нужна popularity normalization, потому что распределение событий по товарам сильно скошено.
- Для товаров без поведения и long-tail товаров нужен fallback.
- Итоговое EDA-решение: переход к MVP preprocessing, sessionization и первому co-visitation baseline допустим; session timeout и long-session caps остаются стартовыми EDA-гипотезами по `sample_day = 2024-04-24` и требуют multi-day проверки.


In [72]:
final_eda_cache_filesystem_metadata = pl.DataFrame([
    cache_filesystem_metadata(path)
    for path in EDA_CACHE_PATHS
])

print("Final EDA cache filesystem metadata:")
display(final_eda_cache_filesystem_metadata)

stale_cache_count = final_eda_cache_filesystem_metadata.filter(
    pl.col("is_stale_vs_raw") == True
).height
missing_cache_count = final_eda_cache_filesystem_metadata.filter(
    pl.col("exists") == False
).height

if stale_cache_count > 0 or missing_cache_count > 0:
    raise RuntimeError(
        f"EDA cache freshness check failed: {stale_cache_count} stale cache(s), "
        f"{missing_cache_count} missing cache(s)."
    )


Final EDA cache filesystem metadata:


path,exists,size_mb,modified_at,is_stale_vs_raw
str,bool,f64,datetime[μs],bool
"""outputs/interim/eda_rows_by_pa…",true,0.003752,2026-05-02 15:48:45.263371,false
"""outputs/interim/eda_partition_…",true,0.015345,2026-05-02 15:48:57.121725,false
"""outputs/interim/eda_user_daily…",true,0.006668,2026-05-02 15:49:24.730509,false
"""outputs/interim/eda_item_actio…",true,19.313365,2026-05-02 15:49:34.513649,false
"""outputs/interim/eda_missing_by…",true,0.016034,2026-05-02 15:51:12.963613,false
"""outputs/interim/eda_duplicate_…",true,0.016799,2026-05-02 15:54:41.951410,false
"""outputs/interim/eda_top_user_b…",true,0.003721,2026-05-02 15:55:40.422431,false
"""outputs/interim/eda_item_actio…",true,1.354181,2026-05-02 15:55:41.292428,false
"""outputs/interim/eda_sample_day…",true,0.157608,2026-05-02 15:55:42.263364,false


## 14. EDA conclusions

### 1. Product information

- В `product_information` найдено 130 035 товаров.
- `item_id` уникален и может использоваться как основной ключ товара.
- `category_id` заполнен полностью и подходит как основной идентификатор категории внутри текущего датасета.
- `category_name` не является уникальным идентификатором категории: одно название может соответствовать нескольким `category_id`.
- Для fallback-логики лучше использовать `category_id`, а `category_name`, `name`, `brand`, `type` — для интерпретации результатов.
- У бренда есть пропуски и специальное значение `"Нет бренда"`, поэтому такие значения нужно трактовать как неизвестный бренд.

### 2. User actions

- В `user_actions` найдено 561 836 992 событий.
- Данные покрывают период с 2024-03-01 по 2024-04-30, всего 61 день.
- Для каждого дня есть все 5 ожидаемых action-type партиций.
- `date` совпадает с датой из `timestamp`, поэтому daily pipeline можно строить по partition date.
- Датасет большой, поэтому для EDA выбран partition-level подход с lazy scan и кэшируемыми агрегатами.

### 3. Action types

- Найдены все ожидаемые события: `view`, `search`, `click`, `favorite`, `to_cart`.
- Для первого item-to-item co-visitation baseline используем только товарные события: `view`, `click`, `favorite`, `to_cart`.
- `search` не включаем в item-to-item graph, потому что он описывает поисковый запрос, а не прямое товарное взаимодействие.
- Стартовые веса событий: `view = 1.0`, `click = 2.0`, `favorite = 2.5`, `to_cart = 4.0`.

### 4. Missing values

- В `user_id`, `timestamp`, `action_type` пропусков нет.
- В товарных событиях `view`, `click`, `favorite`, `to_cart` пропусков `item_id` нет.
- У `search` `item_id` отсутствует в 100% строк, что ожидаемо.
- У `search` есть небольшая доля пустых или null `search_query`; такие строки нужно исключать из будущего co-search анализа.
- Для item-to-item baseline удаление товарных событий по missing values не требуется.

### 5. Duplicates

- Диагностика точных дубликатов по ключу raw-события была выполнена для `sample_day = 2024-04-24`, а также отдельно по каждой из всех 305 parquet-партиций.
- На `sample_day` проверка выявила 4 557 лишних строк-дубликатов, что составляет около 0,0613% строк за этот день.
- Проверка по партициям выявила 588 391 лишнюю строку-дубликат, что составляет около 0,1047% всех событий; группы дубликатов присутствуют в 299 из 305 партиций.
- Доля дубликатов небольшая, но их наличие подтверждено. Они могут слегка завышать popularity, веса событий и pair counts.
- Перед построением co-visitation graph на этапе preprocessing нужно принять решение, удалять ли точные дубликаты raw events.
- Cross-partition проверка точных дубликатов в EDA Step 4 не выполнялась.

### 6. Users

- Найден выраженный аномальный пользователь: `user_id = 6195822`.
- Этот пользователь является самым активным пользователем во все 61 день.
- Такой паттерн выглядит как технический или аномальный трафик.
- Перед построением co-visitation graph нужно исключить или ограничить вклад этого пользователя.
- На EDA-этапе физически строки не удалялись; решение фиксируется для preprocessing.

### 7. Items

- В товарных событиях найдено 161 218 уникальных `item_id`.
- В каталоге есть 130 035 товаров, из них 72 793 имеют поведенческие события.
- 57 242 товара из каталога не имеют поведения; для них нужен fallback.
- В логах есть 88 425 `item_id`, которых нет в каталоге. Их много по количеству, но на них приходится только около 0.64% товарных событий.
- Для MVP товары вне каталога нужно исключить из recommendation candidates.
- Распределение событий по товарам сильно скошено, поэтому raw pair count будет усиливать популярные товары.
- Для baseline нужна popularity normalization.

### 8. Search queries

- `search_query` почти всегда заполнен: пригодны около 99.74% search-событий.
- После нормализации найдено 2 761 580 уникальных запросов.
- Распределение запросов сильно long-tail: большинство запросов встречается очень редко.
- Топ запросов выглядят как реальные пользовательские интенты.
- На `sample_day = 2024-04-24` после исключения аномального пользователя `6195822` около 94.93% `click/to_cart` событий имеют предшествующее search-событие того же пользователя в 30-минутном окне. Это диагностическая временная связь, а не доказательство причинности.
- Co-search не включаем в первый baseline, но оставляем как перспективное улучшение.

### 9. Sessions

- Глобально проверено, что `timestamp` распарсен и согласован с partition `date`; на `sample_day = 2024-04-24` после сортировки по `user_id` и `timestamp` события можно использовать для временного упорядочивания.
- Порядок внутри одинакового timestamp не гарантирован и должен учитываться как ограничение; для co-visitation это допустимо, потому что строгий порядок действий внутри секунды не используется.
- `session_timeout_minutes = 30` — стартовая гипотеза по `sample_day = 2024-04-24`, а не глобально валидированный порог.
- На `sample_day = 2024-04-24` найдена большая доля одинаковых timestamp и длинный хвост по размеру сессий: некоторые сессии содержат слишком много событий и уникальных товаров.
- По sample-day диагностике нужно предусмотреть механизм ограничения длинных сессий или генерации пар внутри них; конкретные пороги требуют проверки на нескольких днях.

### 10. Preprocessing decisions

- Удалять строки с пустыми `user_id`, `timestamp`, `action_type`, если такие появятся.
- Исключить `search` из item-to-item graph и обрабатывать отдельно.
- Исключить или ограничить вклад `user_id = 6195822`.
- Исключить товары вне `product_information` из recommendation candidates. Behavioral evidence for these items is a separate preprocessing decision and should not be dropped automatically in EDA.
- Использовать fallback для товаров без поведения и long-tail товаров.
- Ограничивать длинные сессии по числу событий или уникальных товаров; текущие пороги требуют проверки за пределами `sample_day = 2024-04-24`.
- Нормализовать score по популярности товаров.

### 11. Baseline decisions

- Первый baseline: weighted co-visitation по сессиям.
- Используем события: `view`, `click`, `favorite`, `to_cart`.
- Не используем `search` в первом item-to-item graph.
- Стартовый session timeout: 30 минут как MVP-гипотеза по `sample_day = 2024-04-24`; проверить на multi-day перед финализацией.
- Нужны ограничения: anomalous users и out-of-catalog items подтверждены глобальными/aggregate проверками; long-session caps основаны на sample-day диагностике и требуют проверки порогов.
- Нужна popularity normalization, например cosine-like формула:

```text
score(i, j) = pair_score(i, j) / sqrt(popularity(i) * popularity(j))
```
